In [1]:
import os
import random
import sys
import cv2
import tqdm
import json
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import multilabel_confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [2]:
!pip install polars timm

import polars as pl

In [3]:
!nvidia-smi

Sat May  9 11:18:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               On  |   00000000:4F:00.0 Off |                  Off |
| 38%   66C    P2            217W /  230W |   11197MiB /  24564MiB |     86%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
os.chdir('/workspace')

In [5]:
import pandas as pd
import os

BASE_PATH = 'vindr_mammogram'
meta_path = os.path.join(BASE_PATH, 'metadata.csv')

device = pd.read_csv(meta_path)


device.head()

,SOP Instance UID,Series Instance UID,SOP Instance UID.1,Patient's Age,View Position,Image Laterality,Photometric Interpretation,Rows,Columns,Imager Pixel Spacing,...,Pixel Padding Value,Pixel Padding Range Limit,Window Center,Window Width,Rescale Intercept,Rescale Slope,Rescale Type,Window Center & Width Explanation,Manufacturer,Manufacturer's Model Name
0,d8125545210c08e1b1793a5af6458ee2,b36517b9cbbcfd286a7ae04f643af97a,d8125545210c08e1b1793a5af6458ee2,053Y,CC,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1662,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
1,290c658f4e75a3f83ec78a847414297c,b36517b9cbbcfd286a7ae04f643af97a,290c658f4e75a3f83ec78a847414297c,053Y,MLO,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1664,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
2,cd0fc7bc53ac632a11643ac4cc91002a,b36517b9cbbcfd286a7ae04f643af97a,cd0fc7bc53ac632a11643ac4cc91002a,053Y,CC,R,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1600,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
3,71638b1e853799f227492bfb08a01491,b36517b9cbbcfd286a7ae04f643af97a,71638b1e853799f227492bfb08a01491,053Y,MLO,R,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1654,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
4,dd9ce3288c0773e006a294188aadba8e,d931832a0815df082c085b6e09d20aac,dd9ce3288c0773e006a294188aadba8e,042Y,CC,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1580,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration


In [6]:
print(device['Series Instance UID'].value_counts().value_counts())

count
4    4972
1      29
2      22
3      13
Name: count, dtype: int64


In [7]:
mask = device['Series Instance UID'].value_counts() != 4
print(device['Series Instance UID'].value_counts()[mask])

Series Instance UID
306cd45eecb913f10ce8edb80c3bc70e    3
667b7dc83aec3111ac700cf7f4812838    3
832ed2ef19c22075bffd17826981c49a    3
7b0162b7e4aa6672aa3034cc3fae5e3b    3
63fc20550753b6e8a249422bbbafacc9    3
                                   ..
dda45e7f7e76c34ac6134c05badc577b    1
269abeb40abfed98efdc989e265f5554    1
3704207e1b40779cca4d3de6c66b186b    1
d701ea42302215d0ed0473addd8c2661    1
35c90148a6a93dc6d4114937317a409b    1
Name: count, Length: 64, dtype: int64


In [8]:
mask.head()

Series Instance UID
b36517b9cbbcfd286a7ae04f643af97a    False
d931832a0815df082c085b6e09d20aac    False
a78f4822d806b4f69ba9f0e0c68778b4    False
ca7f2ada530075dd3cf15df6ee51c835    False
bc183447730d58709da1af503d7c469c    False
Name: count, dtype: bool

In [9]:
device.shape

(20000, 21)

In [10]:
df = pl.read_csv("vindr_mammogram/mammo_processed_merged1/mammo_processed_merged.csv")
df = df.to_pandas()
df["merged_image_path"] = (
    df["merged_image_path"]
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
)

In [11]:
density_mapping = {'A': 0, 'B': 0, 'C': 1, 'D': 1}

# density_mapping = {'A': 0, 'B': 0, 'C': 1, 'D': 1}
df['density'] = df['cc_breast_density'].str[-1].map(density_mapping)

In [12]:
import pandas as pd

device_info = device[['SOP Instance UID', "Manufacturer's Model Name"]].copy()
device_info.columns = ['image_id', 'device_model']

df = df.merge(device_info, left_on='cc_image_id', right_on='image_id', how='left')
df = df.drop('image_id', axis=1)

device_mapping = {
    'Mammomat Inspiration': 0,
    'Planmed Nuance': 1,
    'GIOTTO CLASS': 2,
    'GIOTTO IMAGE 3DL': 2
}

df['device_id'] = df['device_model'].map(device_mapping)

print(df['device_model'].value_counts())
print(df['device_id'].value_counts())
print(df.shape)

device_model
Mammomat Inspiration    7621
Planmed Nuance          1898
GIOTTO CLASS             314
GIOTTO IMAGE 3DL         166
Name: count, dtype: int64
device_id
0    7621
1    1898
2     480
Name: count, dtype: int64
(9999, 27)


In [13]:
def get_combined_finding_6class(cc_findings, mlo_findings, cc_birads, mlo_birads):
    if isinstance(cc_findings, str):
        cc_findings = eval(cc_findings) if cc_findings else []
    if isinstance(mlo_findings, str):
        mlo_findings = eval(mlo_findings) if mlo_findings else []
    
    cc_findings = cc_findings or []
    mlo_findings = mlo_findings or []
    all_findings = set(cc_findings + mlo_findings)
    
    if len(all_findings) > 1 and 'No Finding' in all_findings:
        all_findings.remove('No Finding')
    
    high_suspicion_structural = {
        'Architectural Distortion',
        'Skin Thickening',
        'Skin Retraction',
        'Nipple Retraction'
    }
    
    asymmetry_findings = {
        'Focal Asymmetry',
        'Global Asymmetry',
        'Asymmetry'
    }
    
    has_mass = 'Mass' in all_findings
    has_calc = 'Suspicious Calcification' in all_findings
    has_structural = bool(all_findings & high_suspicion_structural)
    has_asymmetry = bool(all_findings & asymmetry_findings)
    has_lymph = 'Suspicious Lymph Node' in all_findings
    
    def parse_birads(birads_str):
        if pd.isna(birads_str) or birads_str == '':
            return 0
        if isinstance(birads_str, str):
            try:
                return int(birads_str.strip().split()[-1])
            except:
                return 0
        return int(birads_str)
    
    cc_birads_num = parse_birads(cc_birads)
    mlo_birads_num = parse_birads(mlo_birads)
    max_birads = max(cc_birads_num, mlo_birads_num)
    
    if not all_findings or all_findings == {'No Finding'}:
        if max_birads == 1:
            return 0
        elif max_birads == 2:
            return 1
        else:
            return 1 if max_birads == 3 else 4
    
    if (has_mass and has_calc) or has_lymph:
        return 4  # Complex/Combined
    
    # Priority 5: Architectural distortions (without mass)
    if has_structural:
        return 5
    
    # Priority 3: Mass (isolated)
    if has_mass:
        return 3
    
    # Priority 2: Calcification (isolated)
    if has_calc:
        return 2
    
    if has_asymmetry and len(all_findings) == 1:
        return -1
    
    if has_asymmetry and len(all_findings) > 1:
        return 5
    
    print(f"Warning: Unknown finding combination: {all_findings}, BIRADS: {max_birads}")
    return 5

df['finding'] = df.apply(
    lambda row: get_combined_finding_6class(
        row['cc_finding_categories'], 
        row['mlo_finding_categories'],
        row['cc_breast_birads'],
        row['mlo_breast_birads']
    ),
    axis=1
)
df.drop(df[df['finding'] == -1].index, inplace=True)
df['finding'].value_counts().sort_index()

finding
0    6703
1    2329
2     127
3     483
4      85
5      83
Name: count, dtype: int64

In [14]:
import ast
import pandas as pd
from collections import Counter

ASYMMETRY_FINDINGS  = frozenset({"Asymmetry", "Focal Asymmetry", "Global Asymmetry"})
STRUCTURAL_FINDINGS = frozenset({"Architectural Distortion", "Skin Thickening", "Skin Retraction", "Nipple Retraction"})
FINDING_TO_BINARY   = [0, 1, 1, 1, 1, 1]
NUM_PROTOTYPES      = 6

def _parse_findings(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return frozenset()
    if isinstance(raw, (list, tuple, set)):
        return frozenset(str(f).strip() for f in raw if str(f).strip())
    if isinstance(raw, str):
        raw = raw.strip()
        if not raw:
            return frozenset()
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, (list, tuple, set)):
                return frozenset(str(f).strip() for f in parsed if str(f).strip())
        except (ValueError, SyntaxError):
            pass
        return frozenset({raw})
    return frozenset()

def _parse_birads(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return 0
    if isinstance(raw, (int, float)):
        return int(raw)
    s = str(raw).strip()
    for token in reversed(s.split()):
        digits = "".join(c for c in token if c.isdigit())
        if digits:
            return int(digits[0])
    return 0

def add_finding_columns(df,
                        cc_findings_col="cc_finding_categories",
                        mlo_findings_col="mlo_finding_categories",
                        cc_birads_col="cc_breast_birads",
                        mlo_birads_col="mlo_breast_birads"):
    rows, drop_idx, other_log = [], [], []

    for idx, row in df.iterrows():
        cc       = _parse_findings(row[cc_findings_col])
        mlo      = _parse_findings(row[mlo_findings_col])
        combined = cc | mlo

        if len(combined) > 1 and "No Finding" in combined:
            combined = combined - {"No Finding"}

        non_asym = combined - ASYMMETRY_FINDINGS
        if combined and not non_asym:
            drop_idx.append(idx)
            continue
        combined = non_asym

        max_birads  = max(_parse_birads(row[cc_birads_col]), _parse_birads(row[mlo_birads_col]))
        is_negative = not combined or combined == {"No Finding"}

        KNOWN = {"No Finding", "Mass", "Suspicious Calcification", "Suspicious Lymph Node"}
        remaining = combined - KNOWN - STRUCTURAL_FINDINGS

        if not is_negative and remaining:
            other_log.extend(list(remaining))

        rows.append({
            "idx":                idx,
            "has_neg_b1":         int(is_negative and max_birads <= 1),
            "has_neg_b2":         int(is_negative and max_birads > 1),
            "has_mass":           int("Mass" in combined),
            "has_calc":           int("Suspicious Calcification" in combined),
            "has_structural":     int(bool(combined & STRUCTURAL_FINDINGS)),
            "has_lymph": int(not is_negative and (
                                      "Suspicious Lymph Node" in combined or bool(remaining))),
        })

    print("=== Top findings landing in has_lymph_or_other ===")
    for finding, count in Counter(other_log).most_common(20):
        print(f"  {finding}: {count}")

    encoded = pd.DataFrame(rows).set_index("idx")
    df = df.drop(index=drop_idx).join(encoded).reset_index(drop=True)
    return df

df = add_finding_columns(df)

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph"]
print("\n=== Finding counts ===")
print(df[FINDING_COLS].sum())


=== Top findings landing in has_lymph_or_other ===

=== Finding counts ===
has_neg_b1        6703
has_neg_b2        2329
has_mass           570
has_calc           207
has_structural      90
has_lymph           22
dtype: int64


In [15]:
wwwwwwwww

NameError: name 'wwwwwwwww' is not defined

In [ ]:
# Count of each finding per device
cross_tab = pd.crosstab(
    df['finding'], 
    df['device_id'], 
    margins=True, 
    margins_name='Total'
)
# Rename columns for clarity
device_names = {0: 'Mammomat', 1: 'Planmed', 2: 'GIOTTO', 'Total': 'Total'}
cross_tab.columns = [device_names.get(c, c) for c in cross_tab.columns]
print("=== Finding Distribution per Device (Counts) ===")
print(cross_tab)

# Percentage within each device (column-wise %)
cross_tab_pct = pd.crosstab(
    df['finding'], 
    df['device_id'], 
    normalize='columns'
).round(3) * 100
cross_tab_pct.columns = [device_names.get(c, c) for c in cross_tab_pct.columns]
print("\n=== Finding Distribution per Device (% of device) ===")
print(cross_tab_pct)

# Percentage within each finding (row-wise %)
cross_tab_row_pct = pd.crosstab(
    df['finding'], 
    df['device_id'], 
    normalize='index'
).round(3) * 100
cross_tab_row_pct.columns = [device_names.get(c, c) for c in cross_tab_row_pct.columns]
print("\n=== Device Distribution per Finding (% of finding) ===")
print(cross_tab_row_pct)

In [ ]:
# df['finding'] = df.apply(
#     lambda row: get_combined_finding(row['cc_finding_categories'], row['mlo_finding_categories']),
#     axis=1
# )
# print(df['finding'].value_counts())

In [ ]:
inbreast_df = pd.read_csv("inbreast_data/INbreast_merged/merged_metadata.csv")
inbreast_df["merged_image_path"] = (
    inbreast_df["merged_image_path"]
    .str.replace("INbreast Release 1.0", "inbreast_data", regex=False)
    .str.replace("vindr_original_data", "inbreast_data", regex=False))
inbreast_df['birads'] = inbreast_df['birads'].replace({'4a': '4', '4b': '4', '4c': '4','6':'5'})
def birads_to_binary(birads):
    if birads in ['1']:
        return 0
    # if birads in ['2']:
    #     return 1
    else:
        return 1
inbreast_df['label'] = inbreast_df['birads'].apply(birads_to_binary)

inbreast_df['birads'] = (inbreast_df['birads'].astype(int) - 1).astype(int)
inbreast_df['density'] = 0
inbreast_df['finding'] = 0
inbreast_df['cc_breast_birads'] = None
inbreast_df['mlo_breast_birads'] = None
inbreast_df['cc_breast_density'] = None
inbreast_df['device_id'] = 0
for col in ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph", "has_other"]:
    inbreast_df[col] = 0
inbreast_df.head()

In [ ]:
def birads_to_binary(birads):
    if birads in ['BI-RADS 1']:
        return 0
    # if birads in ['BI-RADS 2']:
    #     return 1
    return 1 
df['label'] = df['cc_breast_birads'].apply(birads_to_binary)

In [ ]:
def birads_to_label(birads_category):
    """Convert BI-RADS categories to numerical labels 0-4 (for 5 classes)"""
    birads_num = int(birads_category.replace(" ", "")[-1])
    return birads_num - 1
df['birads'] = df['cc_breast_birads'].apply(birads_to_label)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
# df = df[df['view_position'] == 'CC']
# df = df[df['laterality'] == 'R']

In [ ]:

data = df[df['split'] == 'training']
test_df = df[df['split'] == 'test']

In [ ]:
print(data['study_id'].value_counts().value_counts())

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

views_per_study    = data.groupby('study_id').size()
complete_studies   = views_per_study[views_per_study == 2].index
incomplete_studies = views_per_study[views_per_study != 2].index

study_meta = (
    data[data['study_id'].isin(complete_studies)]
    .groupby('study_id')
    .agg({
        'cc_breast_birads': 'first',
        'finding':          'first',
        'device_id':        'first'
    })
    .reset_index()
)

study_meta['stratify_key'] = (
    study_meta['cc_breast_birads'].astype(str) + '_' +
    study_meta['finding'].astype(str)           + '_' +
    study_meta['device_id'].astype(str)
)

key_counts = study_meta['stratify_key'].value_counts()
rare_keys  = key_counts[key_counts < 2].index
study_meta['stratify_key'] = study_meta['stratify_key'].apply(
    lambda k: 'rare' if k in rare_keys else k
)

train_studies, val_studies = train_test_split(
    study_meta['study_id'],
    test_size    = 0.1,
    stratify     = study_meta['stratify_key'],
    random_state = 423
)

train_studies = pd.concat([
    pd.Series(train_studies),
    pd.Series(incomplete_studies)
]).reset_index(drop=True)

train_df = data[data['study_id'].isin(train_studies)].copy()
val_df   = data[data['study_id'].isin(val_studies)].copy()

# fix incomplete in train — keep as single view (don't drop)
train_views = train_df.groupby('study_id').size()
if (train_views != 2).any():
    single_view = train_views[train_views != 2].index
    train_df    = train_df[~train_df['study_id'].isin(single_view)].copy()
    single_df   = data[data['study_id'].isin(single_view)].copy()
    train_df    = pd.concat([train_df, single_df], ignore_index=True)

print(f"Train studies: {train_df['study_id'].nunique()} | Train samples: {len(train_df)}")
print(f"Val   studies: {val_df['study_id'].nunique()}   | Val   samples: {len(val_df)}")

In [ ]:
print(train_df.groupby('study_id').size().value_counts())

In [ ]:
test_df['label'].value_counts()

In [ ]:
mean, std, size, bs = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225], 224, 8

In [ ]:

TARGET_SIZE=(512,256)
  

In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import pandas as pd
from PIL import Image
import os
from torchvision import transforms
import matplotlib.pyplot as plt

import random

In [ ]:
import numpy as np
import cv2
from PIL import Image
import torchvision.transforms as transforms
import random
import torch


def get_transforms(img_size=(512, 512)):
    """Enhanced mammogram preprocessing with medical imaging considerations"""
    
    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        
        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=10,
                translate=(0.05, 0.05),
                scale=(0.9, 1.1),
                shear=6
            )
        ], p=0.6),
        
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomApply([
            transforms.RandomPerspective(distortion_scale=0.1, p=1.0)
        ], p=0.1),
        
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=5.0, sigma=3.0)
        ], p=0.2),
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.95, 1.05),
                contrast=(0.9, 1.1)
            )
        ], p=0.6),
        transforms.Lambda(lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2)) 
                         if random.random() < 0.4 else x),
        

        
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
        ], p=0.2),
        
                # NOISE AUGMENTATIONS
        transforms.Lambda(lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.03)) 
                         if random.random() < 0.4 else x),
        
        transforms.Lambda(lambda x: add_speckle_noise(x, std=random.uniform(0.01, 0.03)) 
                         if random.random() < 0.2 else x),
        transforms.ToTensor(),
        
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    return train_transform, val_transform

def add_gaussian_noise(image, mean=0, std=0.02):
    """Gaussian noise - electronic noise in imaging sensors"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(mean, std, img_array.shape)
        noisy_img = np.clip(img_array + noise, 0, 1)
        return Image.fromarray((noisy_img * 255).astype(np.uint8))
    return image


def add_speckle_noise(image, std=0.03):
    """Speckle noise - multiplicative noise common in mammography"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(0, std, img_array.shape)
        noisy_img = img_array + img_array * noise
        return Image.fromarray((np.clip(noisy_img, 0, 1) * 255).astype(np.uint8))
    return image

def adjust_gamma(image, gamma=1.0):
    """
    Gamma correction - handles tissue density variation
    Gamma < 1 = brighter, > 1 = darker
    """
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        gamma_corrected = np.power(img_array, gamma)
        return Image.fromarray((gamma_corrected * 255).astype(np.uint8))
    return image

In [ ]:
import numpy as np
import random
from PIL import Image
import torchvision.transforms as T

# ── Noise helpers (unchanged — domain-correct) ────────────────────────────────
def add_gaussian_noise(image, mean=0, std=0.02):
    """Electronic sensor noise — additive Gaussian"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        noisy = np.clip(arr + np.random.normal(mean, std, arr.shape), 0, 1)
        return Image.fromarray((noisy * 255).astype(np.uint8))
    return image

def add_speckle_noise(image, std=0.03):
    """Multiplicative speckle — physically correct for mammography"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(0, std, arr.shape)
        noisy = np.clip(arr + arr * noise, 0, 1)
        return Image.fromarray((noisy * 255).astype(np.uint8))
    return image

def adjust_gamma(image, gamma=1.0):
    """Gamma correction — tissue density variation simulation"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        corrected = np.power(arr, gamma)
        return Image.fromarray((corrected * 255).astype(np.uint8))
    return image

def random_90_rotation(image):
    """
    Only 90°/180°/270° steps — avoids interpolation artifact on
    microcalcifications (Shia et al. 2024, Hamidinekoo et al. 2017)
    """
    k = random.choice([0, 1, 2, 3])
    return image.rotate(k * 90)

def get_transforms(img_size=(512, 512)):

    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),

        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomPerspective(distortion_scale=0.1, p=0.2),
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=15.0, sigma=4.0)
        ], p=0.25),

        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=5,
                translate=None,
                scale=(0.9, 1.05),
                shear=0,
            )
        ], p=0.5),

        # Contrast + brightness — tissue density varies across patients
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.8, 1.2),
                contrast=(0.7, 1.3),
            )
        ], p=0.5),

        # Gamma — handles exposure variation between machines
        transforms.Lambda(
            lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2))
            if random.random() < 0.4 else x
        ),

        # Gaussian noise — simulates sensor noise, keep it very mild
        transforms.Lambda(
            lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.02))
            if random.random() < 0.3 else x
        ),

        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    return train_transform, val_transform

In [ ]:
import os
import torch
import random
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

Image.MAX_IMAGE_PIXELS = None
torch.manual_seed(2024)

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph"]


class CustomDataset(Dataset):
    def __init__(self, df, transformations=None, ref_transform=None):
        self.df            = df.reset_index(drop=True)
        self.transform     = transformations
        self.ref_transform = ref_transform
        self.cls_names     = {0: "normal", 1: "abnormal"}
        self.cls_counts    = df['label'].value_counts().to_dict()

    def __len__(self):
        return len(self.df)

    def get_cls_name(self, label):
        return self.cls_names[label]

    def get_pos_neg_im_paths(self, qry_idx, qry_label, qry_density, qry_device, qry_finding):
        def _paths(mask):
            return self.df[mask & (self.df.index != qry_idx)]['merged_image_path'].tolist()

        diff_dev  = self.df['device_id']           != qry_device
        same_den  = self.df['cc_breast_density']   == qry_density
        same_find = self.df['finding']             == qry_finding
        base_pos  = self.df['label']               == qry_label
        base_neg  = self.df['label']               != qry_label

        if qry_label == 0:
            pos_paths = (
                _paths(base_pos & diff_dev & same_den) or
                _paths(base_pos & diff_dev)            or
                _paths(base_pos & same_den)            or
                _paths(base_pos)
            )
        else:
            pos_paths = (
                _paths(base_pos & diff_dev & same_find) or
                _paths(base_pos & same_find)
            )

        neg_paths = (
            _paths(base_neg & diff_dev & same_den) or
            _paths(base_neg & diff_dev)            or
            _paths(base_neg & same_den)            or
            _paths(base_neg)
        )

        return random.choice(pos_paths), random.choice(neg_paths)

    def __getitem__(self, idx):
        row         = self.df.iloc[idx]
        qry_label   = row['birads']
        qry_density = row['cc_breast_density']
        qry_finding = row['finding']
        qry_device  = row['device_id']
        qry_im = Image.open(row['merged_image_path']).convert("RGB")
        if self.transform:
            qry_im = self.transform(qry_im)
        finding_vec = torch.tensor(
            [row[col] for col in FINDING_COLS], dtype=torch.float32
        )
    
        # binary: label=0 (BI-RADS 1) → 0, all other labels → 1
        binary_label = 0 if qry_label == 0 else 1
    
        return {
            "qry_im":      qry_im,
            "qry_gt":      torch.tensor(qry_label,    dtype=torch.long),
            "binary":      torch.tensor(binary_label, dtype=torch.long),
            "finding":     qry_finding,
            "finding_vec": finding_vec,
            "has_neg_b1":     torch.tensor(row["has_neg_b1"],     dtype=torch.long),
            "has_neg_b2":     torch.tensor(row["has_neg_b2"],     dtype=torch.long),
            "has_mass":       torch.tensor(row["has_mass"],       dtype=torch.long),
            "has_calc":       torch.tensor(row["has_calc"],       dtype=torch.long),
            "has_structural": torch.tensor(row["has_structural"], dtype=torch.long),
            "has_lymph":      torch.tensor(row["has_lymph"],      dtype=torch.long),
        }


def get_dls(train_df, val_df, test_df, inbreast_df, bs, ns=6):
    train_trans, val_trans = get_transforms(img_size=(512, 512))

    tr_ds       = CustomDataset(train_df,    train_trans, val_trans)
    vl_ds       = CustomDataset(val_df,      val_trans,   val_trans)
    test_ds     = CustomDataset(test_df,     val_trans,   val_trans)
    inbreast_ds = CustomDataset(inbreast_df, val_trans,   val_trans)

    labels                       = train_df['label'].values
    unique_classes, class_counts = np.unique(labels, return_counts=True)
    beta                         = 0.5
    class_weights                = (1.0 / class_counts) ** beta
    class_weights                = class_weights / class_weights.sum() * len(unique_classes)
    sample_weights               = class_weights[labels]

    print("Class counts:",           dict(zip(unique_classes, class_counts)))
    print("Smoothed class weights:", np.round(class_weights, 3))
    print("Device distribution:\n",  train_df['device_id'].value_counts())

    sampler = WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights).float(),
        num_samples = len(sample_weights),
        replacement = True,
    )

    tr_dl = DataLoader(
        tr_ds, batch_size=bs, 
        # shuffle=True,
        sampler=sampler,
        drop_last=True, num_workers=8, pin_memory=True, persistent_workers=True,
    )
    val_dl = DataLoader(
        vl_ds, batch_size=bs, shuffle=False,
        drop_last=False, num_workers=8, pin_memory=True, persistent_workers=True,
    )
    ts_dl       = DataLoader(test_ds,     batch_size=1, shuffle=False, num_workers=ns)
    inbreast_dl = DataLoader(inbreast_ds, batch_size=1, shuffle=False, num_workers=ns)

    return tr_dl, val_dl, ts_dl, inbreast_dl, tr_ds.cls_names, tr_ds.cls_counts

In [ ]:
tr_dl, val_dl, ts_dl, inbreast_dl, classes, cls_counts = get_dls(train_df,val_df, test_df, inbreast_df, bs =16)

In [ ]:
# import os
# import gc
# import random
# import numpy as np

# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torchvision.models as models

# from tqdm import tqdm
# from torch.optim import AdamW
# from torch.optim.lr_scheduler import CosineAnnealingLR
# from sklearn.metrics import (
#     f1_score,
#     accuracy_score,
#     confusion_matrix,
#     classification_report,
# )

# # =============================================================================
# # Config
# # =============================================================================

# FINDING_COLS = [
#     "has_neg_b1",       # finding 0
#     "has_neg_b2",       # finding 1
#     "has_mass",         # finding 2
#     "has_calc",         # finding 3
#     "has_structural",   # finding 4
#     "has_lymph",        # finding 5
# ]

# NUM_FINDINGS       = 6
# NUM_CLASSES        = 5   # BI-RADS 1-5
# BINARY_CLASS_NAMES = ["Negative (BI-RADS 1)", "Positive (BI-RADS 2-5)"]
# BIRADS_CLASS_NAMES = ["BI-RADS 1", "BI-RADS 2", "BI-RADS 3", "BI-RADS 4", "BI-RADS 5"]

# # Sample weights for contrastive losses
# # These are used to weight each sample's loss contribution
# # Rare findings/classes get higher weights
# FINDING_SAMPLE_WEIGHT = torch.tensor([
#     0.1,   # 0: neg_b1      (4824 - majority)
#     0.2,   # 1: neg_b2      (1666)
#     1.5,   # 2: mass        ( 414 - minority)
#     1.5,   # 3: calc        ( 146 - minority)
#     1.5,   # 4: structural  (  72 - minority)
#     1.5,   # 5: lymph       (  18 - very rare)
# ])

# BIRADS_SAMPLE_WEIGHT = torch.tensor([
#     0.1,   # 0: BI-RADS 1 (4824 - majority)
#     0.2,   # 1: BI-RADS 2 (1666)
#     1.5,   # 2: BI-RADS 3 ( ~414 - minority)
#     1.5,   # 3: BI-RADS 4 ( ~270 - minority)
#     1.5,   # 4: BI-RADS 5 (  ~63 - very rare)
# ])


# def seed_everything(seed=42):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)
#     os.environ["PYTHONHASHSEED"] = str(seed)
#     torch.backends.cudnn.deterministic = False
#     torch.backends.cudnn.benchmark = True


# # =============================================================================
# # Attention Pooling
# # =============================================================================

# class AttentionPool2d(nn.Module):
#     def __init__(self, in_channels, reduction=4):
#         super().__init__()
#         h = max(in_channels // reduction, 32)
#         self.attn = nn.Sequential(
#             nn.Conv2d(in_channels, h, kernel_size=1, bias=False),
#             nn.BatchNorm2d(h),
#             nn.GELU(),
#             nn.Conv2d(h, 1, kernel_size=1, bias=True),
#         )

#     def forward(self, x):
#         w = F.softmax(self.attn(x).flatten(2), dim=-1)
#         return (x.flatten(2) * w).sum(-1)


# # =============================================================================
# # Backbone Encoder
# # =============================================================================

# class FindingAwareProtoModel(nn.Module):
#     def __init__(
#         self,
#         backbone_name,
#         backbone,
#         emb_dim=128,
#         dropout=0.4,
#     ):
#         super().__init__()
#         self.backbone_name = backbone_name.lower()
#         self.backbone      = backbone

#         if "efficientnet" in self.backbone_name:
#             self.num_features = backbone.classifier[1].in_features
#             backbone.classifier = nn.Identity()
#             self.is_cnn = True
#         elif "convnext" in self.backbone_name:
#             self.num_features = backbone.classifier[2].in_features
#             backbone.classifier = nn.Identity()
#             self.is_cnn = True
#         elif "resnet" in self.backbone_name:
#             self.num_features = backbone.fc.in_features
#             backbone.fc = nn.Identity()
#             self.is_cnn = True
#         elif "densenet" in self.backbone_name:
#             self.num_features = backbone.classifier.in_features
#             backbone.classifier = nn.Identity()
#             self.is_cnn = True
#         elif "swin" in self.backbone_name:
#             self.num_features = backbone.head.in_features
#             backbone.head = nn.Identity()
#             self.is_cnn = False
#         else:
#             raise ValueError(f"Unsupported backbone: {backbone_name}")

#         self.pool = AttentionPool2d(self.num_features) if self.is_cnn else None

#         # ── Binary classification head ───────────────────────────────────────
#         # ONLY classification output: class 0 = BI-RADS 1, class 1 = BI-RADS 2-5
#         self.binary_head = nn.Sequential(
#             nn.Linear(self.num_features, 256),
#             nn.LayerNorm(256),
#             nn.ReLU(inplace=True),
#             nn.Dropout(dropout),
#             nn.Linear(256, 2),              # [B, 2]
#         )

#         # ── Finding projection head ──────────────────────────────────────────
#         # Produces 128d normalized embeddings for finding contrastive loss
#         self.proj_head_finding = nn.Sequential(
#             nn.Linear(self.num_features, self.num_features),
#             nn.LayerNorm(self.num_features),
#             nn.ReLU(inplace=True),
#             nn.Linear(self.num_features, emb_dim),
#         )

#         # ── BI-RADS projection head ──────────────────────────────────────────
#         # Produces 128d normalized embeddings for ordinal contrastive loss
#         self.proj_head_birads = nn.Sequential(
#             nn.Linear(self.num_features, self.num_features),
#             nn.LayerNorm(self.num_features),
#             nn.ReLU(inplace=True),
#             nn.Linear(self.num_features, emb_dim),
#         )

#         self._init_weights()

#     def _init_weights(self):
#         for head in [self.binary_head, self.proj_head_finding, self.proj_head_birads]:
#             for m in head.modules():
#                 if isinstance(m, nn.Linear):
#                     nn.init.trunc_normal_(m.weight, std=0.02)
#                     if m.bias is not None:
#                         nn.init.zeros_(m.bias)
#                 elif isinstance(m, (nn.LayerNorm, nn.BatchNorm1d)):
#                     if hasattr(m, "weight") and m.weight is not None:
#                         nn.init.ones_(m.weight)
#                     if hasattr(m, "bias") and m.bias is not None:
#                         nn.init.zeros_(m.bias)

#     def _extract(self, x):
#         f = self.backbone(x)
#         if isinstance(f, tuple):
#             f = f[0]
#         if self.is_cnn:
#             if f.ndim == 4:
#                 f = self.pool(f) if self.pool is not None else f.flatten(2).mean(-1)
#             elif f.ndim == 3:
#                 f = f.mean(1)
#         else:
#             if f.ndim == 3:
#                 f = f.mean(1)
#             elif f.ndim == 4:
#                 f = f.flatten(2).mean(-1)
#         return f

#     def forward(self, x, return_embeddings=False):
#         feat         = self._extract(x)
#         binary_logit = self.binary_head(feat)              # [B, 2]

#         if return_embeddings:
#             emb_f = F.normalize(self.proj_head_finding(feat), dim=1)  # [B, 128]
#             emb_b = F.normalize(self.proj_head_birads(feat),  dim=1)  # [B, 128]
#             return binary_logit, emb_f, emb_b

#         return binary_logit


# # =============================================================================
# # Dual-Path Network (NO Prototypes - Using Supervised Contrastive Instead)
# # =============================================================================

# class DualSupContrastiveNet(nn.Module):
#     def __init__(
#         self,
#         backbone_name,
#         backbone_fn,
#         backbone_weights,
#         emb_dim=128,
#         dropout=0.4,
#         num_classes=NUM_CLASSES,
#         num_findings=NUM_FINDINGS,
#         temperature=0.07,
#     ):
#         super().__init__()
#         self.num_classes  = num_classes
#         self.num_findings = num_findings
#         self.temperature  = temperature
#         self.emb_dim      = emb_dim

#         backbone = backbone_fn(weights=backbone_weights)
#         self.encoder = FindingAwareProtoModel(
#             backbone_name=backbone_name,
#             backbone=backbone,
#             emb_dim=emb_dim,
#             dropout=dropout,
#         )

#     def forward_encoder(self, x, return_embeddings=False):
#         return self.encoder(x, return_embeddings=return_embeddings)


# # =============================================================================
# # Finding-Aware Supervised Contrastive Loss
# #
# # Pull together samples with same finding
# # Push apart samples with different findings
# # Weight each sample by finding rarity (FINDING_SAMPLE_WEIGHT)
# # =============================================================================

# class FindingSupContrastiveLoss(nn.Module):
#     def __init__(self, temperature=0.07):
#         super().__init__()
#         self.tau = temperature

#     def forward(self, embeddings, finding_vec, sample_weights):
#         """
#         embeddings:     [B, D] normalized embeddings
#         finding_vec:    [B, 6] multi-hot finding labels
#         sample_weights: [B] or list of weights per sample from FINDING_SAMPLE_WEIGHT
        
#         Same finding → pull together
#         Different finding → push apart
#         """
#         batch_size = embeddings.shape[0]
        
#         if isinstance(sample_weights, (list, tuple)):
#             sample_weights = torch.tensor(sample_weights, device=embeddings.device, dtype=torch.float)
#         elif not isinstance(sample_weights, torch.Tensor):
#             sample_weights = torch.tensor(sample_weights, device=embeddings.device, dtype=torch.float)
        
#         # Compute pairwise similarity [B, B]
#         sim_matrix = torch.mm(embeddings, embeddings.t()) / self.tau
        
#         # Create positive mask: same finding = positive
#         # finding_vec [B, 6] multi-hot
#         positive_mask = torch.mm(finding_vec, finding_vec.t()) > 0.5  # [B, B]
        
#         # Remove self-positive (diagonal)
#         positive_mask.fill_diagonal_(False)
        
#         losses = []
#         valid_weights = []
        
#         for i in range(batch_size):
#             pos_mask = positive_mask[i]
            
#             # Need at least one positive
#             if pos_mask.sum() == 0:
#                 continue
            
#             # Log softmax over all samples
#             log_softmax = F.log_softmax(sim_matrix[i], dim=0)
            
#             # Loss = -mean(log(exp(sim_pos) / sum(exp(sim_all))))
#             # = -mean(log_softmax[pos_mask])
#             loss_i = -log_softmax[pos_mask].mean()
            
#             losses.append(loss_i)
#             valid_weights.append(sample_weights[i].item())
        
#         if len(losses) == 0:
#             return torch.tensor(0.0, device=embeddings.device, requires_grad=False)
        
#         losses = torch.stack(losses)
#         valid_weights = torch.tensor(valid_weights, device=embeddings.device, dtype=torch.float)
        
#         # Weight by sample importance (rare findings get higher weight)
#         weighted_loss = (valid_weights * losses).sum() / (valid_weights.sum() + 1e-8)
        
#         return weighted_loss


# # =============================================================================
# # Ordinal Supervised Contrastive Loss for BI-RADS
# #
# # Same BI-RADS → weight = 1.0 (pull together strongly)
# # Adjacent BI-RADS (e.g., 2 vs 3) → weight = 0.7 (push apart moderately)
# # Far BI-RADS (e.g., 1 vs 5) → weight = 0.0 (push apart strongly as hard negatives)
# #
# # This respects the ordinal structure: nearby classes are softer negatives
# # =============================================================================

# class OrdinalSupContrastiveLoss(nn.Module):
#     def __init__(self, temperature=0.07):
#         super().__init__()
#         self.tau = temperature

#     def forward(self, embeddings, labels, sample_weights):
#         """
#         embeddings:     [B, D] normalized embeddings
#         labels:         [B] int BI-RADS class (0-4)
#         sample_weights: [B] or list of weights per sample from BIRADS_SAMPLE_WEIGHT
        
#         Same class → pull together (weight in exponential: 1.0)
#         Adjacent class (ordinal distance = 1) → softer negative (weight: 0.7)
#         Distant class (ordinal distance >= 2) → hard negative (weight: 0.0)
#         """
#         batch_size = embeddings.shape[0]
        
#         if isinstance(sample_weights, (list, tuple)):
#             sample_weights = torch.tensor(sample_weights, device=embeddings.device, dtype=torch.float)
#         elif not isinstance(sample_weights, torch.Tensor):
#             sample_weights = torch.tensor(sample_weights, device=embeddings.device, dtype=torch.float)
        
#         # Compute pairwise similarity [B, B]
#         sim_matrix = torch.mm(embeddings, embeddings.t()) / self.tau
        
#         # Compute ordinal distances [B, B]
#         # dist[i,j] = |label[i] - label[j]|
#         label_matrix = labels.unsqueeze(0) - labels.unsqueeze(1)  # [B, B]
#         ordinal_dist = torch.abs(label_matrix).float()
        
#         # Ordinal weighting for negatives:
#         # Same class (dist=0): full positive, weight in exponent = 0
#         # Adjacent (dist=1): softer negative, weight in exponent = 0.7
#         # Distant (dist>=2): hard negative, weight in exponent = 0.0
#         # Formula: weight = exp(-ordinal_dist * scale)
#         ordinal_weights = torch.exp(-ordinal_dist * 1.0)  # [B, B]
#         # ordinal_weights[i,i] = 1.0 (will be removed by pos_mask)
#         # ordinal_weights[adjacent] ≈ 0.37
#         # ordinal_weights[distant] ≈ 0.14 to 0.04
        
#         # Create positive mask: same label = positive
#         positive_mask = (labels.unsqueeze(0) == labels.unsqueeze(1))  # [B, B]
        
#         # Remove self-positive (diagonal)
#         positive_mask.fill_diagonal_(False)
        
#         # Apply ordinal weights to similarity matrix
#         # For positives: no change (they're already pulled via exponential)
#         # For negatives: reduce their contribution based on ordinal distance
#         # weighted_sim[i,j] = sim[i,j] * (1 - ordinal_weight) when negative
#         #                   = sim[i,j] when positive
        
#         # Alternative: modify sim directly
#         # sim_weighted = sim * ordinal_weights
#         # This way: same class gets full sim, distant classes get reduced sim
        
#         sim_weighted = sim_matrix * ordinal_weights  # [B, B]
        
#         losses = []
#         valid_weights = []
        
#         for i in range(batch_size):
#             pos_mask = positive_mask[i]
            
#             # Need at least one positive
#             if pos_mask.sum() == 0:
#                 continue
            
#             # Log softmax over all samples with ordinal-weighted similarities
#             log_softmax = F.log_softmax(sim_weighted[i], dim=0)
            
#             # Loss = -mean(log_softmax[pos_mask])
#             loss_i = -log_softmax[pos_mask].mean()
            
#             losses.append(loss_i)
#             valid_weights.append(sample_weights[i].item())
        
#         if len(losses) == 0:
#             return torch.tensor(0.0, device=embeddings.device, requires_grad=False)
        
#         losses = torch.stack(losses)
#         valid_weights = torch.tensor(valid_weights, device=embeddings.device, dtype=torch.float)
        
#         # Weight by sample importance (rare classes get higher weight)
#         weighted_loss = (valid_weights * losses).sum() / (valid_weights.sum() + 1e-8)
        
#         return weighted_loss


# # =============================================================================
# # Asymmetric Focal Loss for Classification
# # =============================================================================

# class AsymmetricFocalLoss(nn.Module):
#     def __init__(self, gamma_pos=1.0, gamma_neg=2.0, alpha=0.6):
#         super().__init__()
#         self.gp = gamma_pos
#         self.gn = gamma_neg
#         self.alpha = alpha

#     def forward(self, logits, targets):
#         targets = targets.long()
#         ce      = F.cross_entropy(logits, targets, reduction="none")
#         pt      = torch.exp(-ce)
#         gamma   = torch.where(targets == 1, torch.full_like(ce, self.gp),
#                               torch.full_like(ce, self.gn))
#         alpha_t = torch.where(targets == 1, torch.full_like(ce, self.alpha),
#                               torch.full_like(ce, 1 - self.alpha))
#         return (alpha_t * (1 - pt) ** gamma * ce).mean()


# # =============================================================================
# # Validation
# # =============================================================================

# @torch.no_grad()
# def validate(model, loader, device, binary_loss_fn, class_names=None):
#     class_names = class_names or BINARY_CLASS_NAMES
#     model.eval()

#     total_loss             = 0.0
#     bin_preds, bin_targets = [], []

#     for batch in loader:
#         imgs      = batch["qry_im"].to(device, non_blocking=True)
#         binary_gt = batch["binary"].to(device, non_blocking=True).long()

#         binary_logit = model.forward_encoder(imgs, return_embeddings=False)
#         total_loss  += binary_loss_fn(binary_logit, binary_gt).item()

#         pred = binary_logit.argmax(1)
#         bin_preds.extend(pred.cpu().numpy())
#         bin_targets.extend(binary_gt.cpu().numpy())

#     acc      = accuracy_score(bin_targets, bin_preds)
#     macro_f1 = f1_score(bin_targets, bin_preds, average="macro",    zero_division=0)
#     wt_f1    = f1_score(bin_targets, bin_preds, average="weighted", zero_division=0)

#     print("\n--- Validation (Binary) ---")
#     print(classification_report(bin_targets, bin_preds, target_names=class_names, zero_division=0))

#     return {
#         "loss":        total_loss / max(len(loader), 1),
#         "acc":         acc,
#         "macro_f1":    macro_f1,
#         "weighted_f1": wt_f1,
#     }


# # =============================================================================
# # Test
# # =============================================================================

# @torch.no_grad()
# def test_model(model, loader, device, save_dir, name="Test", class_names=None):
#     class_names = class_names or BINARY_CLASS_NAMES
#     model.eval()

#     bin_preds, bin_targets = [], []

#     for batch in loader:
#         imgs      = batch["qry_im"].to(device, non_blocking=True)
#         binary_gt = batch["binary"].to(device, non_blocking=True).long()

#         binary_logit = model.forward_encoder(imgs, return_embeddings=False)
#         pred         = binary_logit.argmax(1)

#         bin_preds.extend(pred.cpu().numpy())
#         bin_targets.extend(binary_gt.cpu().numpy())

#     acc      = accuracy_score(bin_targets, bin_preds)
#     macro_f1 = f1_score(bin_targets, bin_preds, average="macro",    zero_division=0)
#     wt_f1    = f1_score(bin_targets, bin_preds, average="weighted", zero_division=0)
#     cm       = confusion_matrix(bin_targets, bin_preds)

#     print(f"\n{'='*70}")
#     print(f"{name} | Acc={acc:.4f}  Macro-F1={macro_f1:.4f}  Weighted-F1={wt_f1:.4f}")
#     print(cm)
#     print(classification_report(bin_targets, bin_preds, target_names=class_names, zero_division=0))
#     print('=' * 70)

#     os.makedirs(save_dir, exist_ok=True)
#     with open(os.path.join(save_dir, f"{name.lower().replace(' ', '_')}_report.txt"), "w") as fh:
#         fh.write(f"{name}\n")
#         fh.write(f"Acc={acc:.4f}  Macro-F1={macro_f1:.4f}  Weighted-F1={wt_f1:.4f}\n\n")
#         fh.write(str(cm) + "\n\n")
#         fh.write(classification_report(bin_targets, bin_preds, target_names=class_names, zero_division=0))

#     return {"acc": acc, "macro_f1": macro_f1, "weighted_f1": wt_f1}


# # =============================================================================
# # Training Loop
# # =============================================================================

# def train_model(
#     model,
#     train_loader,
#     val_loader,
#     device,
#     lr_backbone=5e-5,
#     lr_heads=5e-5,
#     epochs=60,
#     patience=15,
#     save_path="best_model.pth",
#     w_finding=0.3,
#     w_birads=0.3,
#     warmup_epochs=3,
#     ramp_epochs=10,
#     class_names=None,
# ):
#     class_names = class_names or BINARY_CLASS_NAMES
#     os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
#     log_path = save_path.replace(".pth", "_log.txt")

#     # Move sample weights to device
#     finding_weights = FINDING_SAMPLE_WEIGHT.to(device)
#     birads_weights = BIRADS_SAMPLE_WEIGHT.to(device)

#     # Loss functions
#     binary_loss_fn = AsymmetricFocalLoss().to(device)
#     finding_loss_fn = FindingSupContrastiveLoss(temperature=0.07).to(device)
#     birads_loss_fn = OrdinalSupContrastiveLoss(temperature=0.07).to(device)

#     # Optimizer
#     optimizer = AdamW([
#         {
#             "params":       model.encoder.backbone.parameters(),
#             "lr":           lr_backbone,
#             "weight_decay": 0.05,
#         },
#         {
#             "params":       model.encoder.binary_head.parameters(),
#             "lr":           lr_heads,
#             "weight_decay": 0.05,
#         },
#         {
#             "params":       model.encoder.proj_head_finding.parameters(),
#             "lr":           lr_heads,
#             "weight_decay": 0.05,
#         },
#         {
#             "params":       model.encoder.proj_head_birads.parameters(),
#             "lr":           lr_heads,
#             "weight_decay": 0.05,
#         },
#     ])

#     scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
#     scaler    = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

#     best_macro_f1 = 0.0
#     not_improved  = 0

#     for epoch in range(epochs):
#         # Learning rate ramp for contrastive losses
#         if epoch < warmup_epochs:
#             pw_f = 0.0
#             pw_b = 0.0
#         else:
#             ramp = min(1.0, (epoch - warmup_epochs + 1) / max(ramp_epochs, 1))
#             pw_f = w_finding * ramp
#             pw_b = w_birads  * ramp
        
#         model.train()
#         e_binary = e_finding = e_birads = 0.0
#         all_preds   = []
#         all_targets = []

#         pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

#         for batch in pbar:
#             imgs        = batch["qry_im"].to(device, non_blocking=True)
#             labels      = batch["qry_gt"].to(device, non_blocking=True).long()
#             binary_gt   = batch["binary"].to(device, non_blocking=True).long()
#             finding_vec = batch["finding_vec"].to(device, non_blocking=True).float()

#             optimizer.zero_grad(set_to_none=True)

#             with torch.amp.autocast(device_type=device.type, enabled=(scaler is not None)):

#                 # Forward pass
#                 binary_logit, emb_f, emb_b = model.forward_encoder(
#                     imgs, return_embeddings=True
#                 )

#                 # ── Loss 1: Binary classification ────────────────────────────
#                 binary_loss = binary_loss_fn(binary_logit, binary_gt)

#                 # ── Loss 2: Finding supervised contrastive ───────────────────
#                 if pw_f > 0:
#                     # Get sample weights for this batch
#                     batch_finding_weights = []
#                     for i in range(imgs.shape[0]):
#                         active_findings = torch.where(finding_vec[i] > 0.5)[0]
#                         if len(active_findings) > 0:
#                             # Use weight of first active finding
#                             f_idx = active_findings[0].item()
#                             batch_finding_weights.append(finding_weights[f_idx].item())
#                         else:
#                             batch_finding_weights.append(0.0)
                    
#                     finding_loss = finding_loss_fn(
#                         embeddings=emb_f,
#                         finding_vec=finding_vec,
#                         sample_weights=batch_finding_weights,
#                     )
#                 else:
#                     finding_loss = torch.tensor(0.0, device=device)

#                 # ── Loss 3: BI-RADS ordinal supervised contrastive ──────────
#                 if pw_b > 0:
#                     # Get sample weights for this batch
#                     batch_birads_weights = [birads_weights[lb.item()].item() for lb in labels]
                    
#                     birads_loss = birads_loss_fn(
#                         embeddings=emb_b,
#                         labels=labels,
#                         sample_weights=batch_birads_weights,
#                     )
#                 else:
#                     birads_loss = torch.tensor(0.0, device=device)

#                 # ── Total loss ───────────────────────────────────────────────
#                 total_loss = binary_loss + pw_f * finding_loss + pw_b * birads_loss

#             # Backward pass
#             if scaler is not None:
#                 scaler.scale(total_loss).backward()
#                 scaler.unscale_(optimizer)
#                 torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), 1.0)
#                 scaler.step(optimizer)
#                 scaler.update()
#             else:
#                 total_loss.backward()
#                 torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), 1.0)
#                 optimizer.step()

#             # Track predictions
#             pred = binary_logit.argmax(1)
#             all_preds.extend(pred.detach().cpu().numpy())
#             all_targets.extend(binary_gt.detach().cpu().numpy())

#             e_binary  += binary_loss.item()
#             e_finding += finding_loss.item() if isinstance(finding_loss, torch.Tensor) else 0.0
#             e_birads  += birads_loss.item() if isinstance(birads_loss, torch.Tensor) else 0.0

#             pbar.set_postfix(
#                 bin=f"{binary_loss.item():.3f}",
#                 find=f"{finding_loss.item():.3f}" if isinstance(finding_loss, torch.Tensor) else "0.0",
#                 bir=f"{birads_loss.item():.3f}" if isinstance(birads_loss, torch.Tensor) else "0.0",
#                 pwf=f"{pw_f:.2f}",
#                 pwb=f"{pw_b:.2f}",
#             )

#         # Training metrics
#         n              = max(len(train_loader), 1)
#         train_acc      = accuracy_score(all_targets, all_preds)
#         train_macro_f1 = f1_score(all_targets, all_preds, average="macro",    zero_division=0)
#         train_wt_f1    = f1_score(all_targets, all_preds, average="weighted", zero_division=0)

#         print(f"\n{'='*70}")
#         print(
#             f"Epoch {epoch+1}/{epochs}  "
#             f"bin={e_binary/n:.4f}  "
#             f"find={e_finding/n:.4f}  bir={e_birads/n:.4f}  "
#             f"pwf={pw_f:.3f}  pwb={pw_b:.3f}  "
#             f"train_acc={train_acc:.4f}  train_macro_f1={train_macro_f1:.4f}"
#         )
#         print("\n--- Train (Binary) ---")
#         print(classification_report(all_targets, all_preds, target_names=class_names, zero_division=0))

#         # Validation
#         val_metrics = validate(
#             model=model,
#             loader=val_loader,
#             device=device,
#             binary_loss_fn=binary_loss_fn,
#             class_names=class_names,
#         )

#         print(
#             f"Val loss={val_metrics['loss']:.4f}  "
#             f"Val acc={val_metrics['acc']:.4f}  "
#             f"Val macro_f1={val_metrics['macro_f1']:.4f}  "
#             f"Val weighted_f1={val_metrics['weighted_f1']:.4f}"
#         )
#         print('=' * 70)

#         scheduler.step()

#         # Logging
#         with open(log_path, "a") as fh:
#             fh.write(
#                 f"Epoch {epoch+1}  "
#                 f"bin={e_binary/n:.4f}  "
#                 f"find={e_finding/n:.4f}  bir={e_birads/n:.4f}  "
#                 f"pwf={pw_f:.3f}  pwb={pw_b:.3f}  "
#                 f"train_acc={train_acc:.4f}  train_macro_f1={train_macro_f1:.4f}  "
#                 f"train_wt_f1={train_wt_f1:.4f}  "
#                 f"val_loss={val_metrics['loss']:.4f}  "
#                 f"val_acc={val_metrics['acc']:.4f}  "
#                 f"val_macro_f1={val_metrics['macro_f1']:.4f}  "
#                 f"val_wt_f1={val_metrics['weighted_f1']:.4f}\n"
#             )

#         # Save best model
#         if val_metrics["macro_f1"] > best_macro_f1:
#             best_macro_f1 = val_metrics["macro_f1"]
#             torch.save({
#                 "epoch":            epoch,
#                 "model_state_dict": model.state_dict(),
#                 "optimizer_state":  optimizer.state_dict(),
#                 "best_macro_f1":    best_macro_f1,
#             }, save_path)
#             print(f"Saved best model macro-F1={best_macro_f1:.4f}")
#             not_improved = 0
#         else:
#             not_improved += 1
#             if not_improved >= patience:
#                 print(f"Early stopping at epoch {epoch+1}")
#                 break

#     return best_macro_f1


# # =============================================================================
# # Main Training Runner
# # =============================================================================

# def run_experiments(
#     train_loader,
#     val_loader,
#     test_loader,
#     inbreast_loader,
# ):
#     seed_everything(42)
#     device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#     print(f"Device: {device}")

#     train_config = dict(
#         lr_backbone   = 5e-5,
#         lr_heads      = 5e-5,
#         epochs        = 60,
#         patience      = 15,
#         w_finding     = 0.5,
#         w_birads      = 0.5,
#         warmup_epochs = 3,
#         ramp_epochs   = 5,
#         class_names   = BINARY_CLASS_NAMES,
#     )

#     backbones = [
#         {
#             "name": "efficientnet_b3",
#             "fn":   models.efficientnet_b3,
#             "w":    models.EfficientNet_B3_Weights.DEFAULT,
#         },
#         {
#             "name": "convnext_base",
#             "fn":   models.convnext_base,
#             "w":    models.ConvNeXt_Base_Weights.DEFAULT,
#         },
#     ]

#     all_results = {}

#     for cfg in backbones:
#         out_dir   = f"/workspace/Thesis_updated_results/binary_ordinal_supcontrastive/{cfg['name']}"
#         save_path = f"{out_dir}/best_model.pth"
#         os.makedirs(out_dir, exist_ok=True)

#         print(f"\n{'#'*70}")
#         print(f"Backbone: {cfg['name']}")
#         print(f"{'#'*70}")
#         print(f"Finding sample weights:  {FINDING_SAMPLE_WEIGHT.tolist()}")
#         print(f"BI-RADS sample weights:  {BIRADS_SAMPLE_WEIGHT.tolist()}")
#         print("Loss: Binary Classification + Finding SupContrastive + Ordinal BI-RADS SupContrastive")

#         model = DualSupContrastiveNet(
#             backbone_name    = cfg["name"],
#             backbone_fn      = cfg["fn"],
#             backbone_weights = cfg["w"],
#             emb_dim          = 128,
#             dropout          = 0.2,
#             num_classes      = NUM_CLASSES,
#             num_findings     = NUM_FINDINGS,
#             temperature      = 0.07,
#         ).to(device)

#         num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
#         print(f"Trainable params: {num_params:,}")

#         best_macro_f1 = train_model(
#             model=model,
#             train_loader=train_loader,
#             val_loader=val_loader,
#             device=device,
#             save_path=save_path,
#             **train_config,
#         )

#         # Load best model
#         ckpt = torch.load(save_path, map_location=device)
#         model.load_state_dict(ckpt["model_state_dict"])
#         print(f"Loaded epoch {ckpt['epoch']+1} | best macro-F1={ckpt['best_macro_f1']:.4f}")

#         # Test on VinDr
#         vindr_metrics = test_model(
#             model=model,
#             loader=test_loader,
#             device=device,
#             save_dir=out_dir,
#             name="VinDr-Test",
#             class_names=BINARY_CLASS_NAMES,
#         )

#         # Test on INbreast
#         inbreast_metrics = test_model(
#             model=model,
#             loader=inbreast_loader,
#             device=device,
#             save_dir=out_dir,
#             name="INbreast",
#             class_names=BINARY_CLASS_NAMES,
#         )

#         all_results[cfg["name"]] = dict(
#             best_val_macro_f1    = best_macro_f1,
#             vindr_acc            = vindr_metrics["acc"],
#             vindr_macro_f1       = vindr_metrics["macro_f1"],
#             vindr_weighted_f1    = vindr_metrics["weighted_f1"],
#             inbreast_acc         = inbreast_metrics["acc"],
#             inbreast_macro_f1    = inbreast_metrics["macro_f1"],
#             inbreast_weighted_f1 = inbreast_metrics["weighted_f1"],
#         )

#         del model, ckpt
#         torch.cuda.empty_cache()
#         gc.collect()

#     # Print final results
#     print(f"\n{'='*85}")
#     print("FINAL RESULTS")
#     print(f"{'='*85}")
#     print(
#         f"{'Model':<22} "
#         f"{'Val Macro-F1':>12} "
#         f"{'VinDr Macro-F1':>15} "
#         f"{'INbreast Macro-F1':>18}"
#     )
#     print("-" * 85)
#     for name, r in all_results.items():
#         print(
#             f"{name:<22} "
#             f"{r['best_val_macro_f1']:>12.4f} "
#             f"{r['vindr_macro_f1']:>15.4f} "
#             f"{r['inbreast_macro_f1']:>18.4f}"
#         )

#     return all_results


# results = run_experiments(
#     train_loader    = tr_dl,
#     val_loader      = val_dl,
#     test_loader     = ts_dl,
#     inbreast_loader = inbreast_dl,
# )

In [ ]:
import os
import gc
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from tqdm import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)


FINDING_COLS = [
    "has_neg_b1",
    "has_neg_b2",
    "has_mass",
    "has_calc",
    "has_structural",
    "has_lymph",
]

NUM_FINDINGS = 6
NUM_CLASSES = 5

BINARY_CLASS_NAMES = ["Negative (BI-RADS 1)", "Positive (BI-RADS 2-5)"]
BIRADS_CLASS_NAMES = ["BI-RADS 1", "BI-RADS 2", "BI-RADS 3", "BI-RADS 4", "BI-RADS 5"]

BINARY_CLASS_WEIGHT = [
    7056.0 / (2.0 * 4824.0),
    7056.0 / (2.0 * 2232.0),
]


BIRADS_PROTO_MOMENTUM = [0.99, 0.99, 0.97, 0.97, 0.95]
FINDING_PROTO_MOMENTUM = [0.99, 0.99, 0.97, 0.98, 0.95, 0.95]


FINDING_SAMPLE_WEIGHT = [0.1, 0.2, 1.2, 1.2, 1.5, 1.5]
BIRADS_SAMPLE_WEIGHT = [0.1, 0.2, 1.00, 1.20, 1.50]
PROTO_MIN_UPDATES = 10


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


class ProtoInfoNCELoss(nn.Module):
    def __init__(self, temperature=0.07, min_updates=PROTO_MIN_UPDATES):
        super().__init__()
        self.tau = temperature
        self.min_updates = min_updates

    def forward(self, q, prototypes, proto_updates, sample_weights_list, pos_indices):
        ready = proto_updates >= self.min_updates
        losses = []
        weights = []

        for i in range(q.shape[0]):
            pos_idx = int(pos_indices[i])
            if pos_idx < 0 or not ready[pos_idx]:
                continue

            qi = q[i]
            sims = []
            pos_local_idx = None

            for p in range(prototypes.shape[0]):
                if not ready[p]:
                    continue

                sim = torch.dot(qi, prototypes[p]) / self.tau

                if p == pos_idx:
                    pos_local_idx = len(sims)

                sims.append(sim)

            if pos_local_idx is None or len(sims) < 2:
                continue

            sims = torch.stack(sims)
            loss_i = -F.log_softmax(sims, dim=0)[pos_local_idx]

            losses.append(loss_i)
            weights.append(float(sample_weights_list[i]))

        if len(losses) == 0:
            return torch.tensor(0.0, device=q.device, dtype=q.dtype)

        losses = torch.stack(losses)
        weights = torch.tensor(weights, device=q.device, dtype=q.dtype)

        return (weights * losses).sum() / (weights.sum() + 1e-8)


class MultiLabelProtoLoss(nn.Module):
    def __init__(self, temperature=0.07, min_updates=PROTO_MIN_UPDATES):
        super().__init__()
        self.tau = temperature
        self.min_updates = min_updates

    def forward(self, q, prototypes, proto_updates, sample_weights_list, finding_vec):
        ready = proto_updates >= self.min_updates
        losses = []
        weights = []

        for i in range(q.shape[0]):
            active_findings = torch.where(finding_vec[i] > 0.5)[0]
            
            if len(active_findings) == 0:
                continue

            qi = q[i]
            
            for finding_idx in active_findings:
                finding_idx = finding_idx.item()
                
                if not ready[finding_idx]:
                    continue

                all_sims = []
                pos_local_idx = None

                for p in range(prototypes.shape[0]):
                    if not ready[p]:
                        continue

                    sim = torch.dot(qi, prototypes[p]) / self.tau

                    if p == finding_idx:
                        pos_local_idx = len(all_sims)

                    all_sims.append(sim)

                if pos_local_idx is None or len(all_sims) < 2:
                    continue

                all_sims = torch.stack(all_sims)
                log_softmax = F.log_softmax(all_sims, dim=0)
                loss_i = -log_softmax[pos_local_idx]

                losses.append(loss_i)
                weights.append(sample_weights_list[i])

        if len(losses) == 0:
            return torch.tensor(0.0, device=q.device, dtype=q.dtype)

        losses = torch.stack(losses)
        weights = torch.tensor(weights, device=q.device, dtype=q.dtype)

        return (weights * losses).sum() / (weights.sum() + 1e-8)


class PairwiseSeparationLoss(nn.Module):
    def __init__(self, margin=0.5, min_updates=10):
        super().__init__()
        self.margin = margin
        self.min_updates = min_updates

    def forward(
        self,
        proto_finding,
        proto_birads,
        proto_finding_updates=None,
        proto_birads_updates=None,
    ):
        loss = proto_finding.new_tensor(0.0)
        count = 0

        for j in [1, 2, 3, 4]:
            if proto_birads_updates is not None:
                if proto_birads_updates[0] < self.min_updates or proto_birads_updates[j] < self.min_updates:
                    continue

            sim = torch.dot(proto_birads[0], proto_birads[j])
            loss += F.relu(sim - self.margin)
            count += 1

        # for j in [2, 3, 4]:
        #     if proto_birads_updates is not None:
        #         if proto_birads_updates[1] < self.min_updates or proto_birads_updates[j] < self.min_updates:
        #             continue

        #     sim = torch.dot(proto_birads[1], proto_birads[j])
        #     loss += F.relu(sim - self.margin)
        #     count += 1

        for j in [1, 2, 3, 4, 5]:
            if proto_finding_updates is not None:
                if proto_finding_updates[0] < self.min_updates or proto_finding_updates[j] < self.min_updates:
                    continue

            sim = torch.dot(proto_finding[0], proto_finding[j])
            loss += F.relu(sim - self.margin)
            count += 1

        # for j in [2, 3, 4, 5]:
        #     if proto_finding_updates is not None:
        #         if proto_finding_updates[1] < self.min_updates or proto_finding_updates[j] < self.min_updates:
        #             continue

        #     sim = torch.dot(proto_finding[1], proto_finding[j])
        #     loss += F.relu(sim - self.margin)
        #     count += 1

        if count == 0:
            return proto_finding.new_tensor(0.0)

        return loss / count


class AsymmetricFocalLoss(nn.Module):
    def __init__(self, gamma_pos=1.0, gamma_neg=2.0, alpha=0.6):
        super().__init__()
        self.gp = gamma_pos
        self.gn = gamma_neg
        self.alpha = alpha

    def forward(self, logits, targets):
        targets = targets.long()
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)

        gamma = torch.where(
            targets == 1,
            torch.full_like(ce, self.gp),
            torch.full_like(ce, self.gn),
        )

        alpha_t = torch.where(
            targets == 1,
            torch.full_like(ce, self.alpha),
            torch.full_like(ce, 1.0 - self.alpha),
        )

        return (alpha_t * (1.0 - pt) ** gamma * ce).mean()


class FindingBiradsProtoEncoder(nn.Module):
    def __init__(self, backbone_name, backbone, emb_dim=128, dropout=0.2):
        super().__init__()
        self.backbone_name = backbone_name.lower()

        if "efficientnet" in self.backbone_name:
            self.features = backbone.features
            self.avgpool = backbone.avgpool
            self.num_features = backbone.classifier[1].in_features
        elif "convnext" in self.backbone_name:
            self.features = backbone.features
            self.avgpool = backbone.avgpool
            self.num_features = backbone.classifier[2].in_features
        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

        self.binary_head = nn.Sequential(
            nn.Linear(self.num_features, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 2),
        )

        self.proj_head_finding = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

        self.proj_head_birads = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

        self._init_weights()

    def _init_weights(self):
        heads = [self.binary_head, self.proj_head_finding, self.proj_head_birads]

        for head in heads:
            for m in head.modules():
                if isinstance(m, nn.Linear):
                    nn.init.trunc_normal_(m.weight, std=0.02)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
                elif isinstance(m, (nn.LayerNorm, nn.BatchNorm1d)):
                    if hasattr(m, "weight") and m.weight is not None:
                        nn.init.ones_(m.weight)
                    if hasattr(m, "bias") and m.bias is not None:
                        nn.init.zeros_(m.bias)

    def _extract(self, x):
        f = self.features(x)
        f = self.avgpool(f)
        f = torch.flatten(f, 1)
        return f

    def forward(self, x, return_embeddings=False):
        feat = self._extract(x)
        binary_logits = self.binary_head(feat)

        if return_embeddings:
            emb_f = F.normalize(self.proj_head_finding(feat), dim=1)
            emb_b = F.normalize(self.proj_head_birads(feat), dim=1)
            return binary_logits, emb_f, emb_b

        return binary_logits


class DualProtoNet(nn.Module):
    def __init__(
        self,
        backbone_name,
        backbone_fn,
        backbone_weights,
        emb_dim=128,
        dropout=0.2,
        num_classes=NUM_CLASSES,
        num_findings=NUM_FINDINGS,
        temperature=0.12,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.num_findings = num_findings
        self.temperature = temperature
        self.emb_dim = emb_dim

        backbone = backbone_fn(weights=backbone_weights)

        self.encoder = FindingBiradsProtoEncoder(
            backbone_name=backbone_name,
            backbone=backbone,
            emb_dim=emb_dim,
            dropout=dropout,
        )

        self.register_buffer(
            "proto_finding",
            F.normalize(torch.randn(num_findings, emb_dim), dim=-1),
        )
        self.register_buffer(
            "proto_finding_updates",
            torch.zeros(num_findings, dtype=torch.long),
        )

        self.register_buffer(
            "proto_birads",
            F.normalize(torch.randn(num_classes, emb_dim), dim=-1),
        )
        self.register_buffer(
            "proto_birads_updates",
            torch.zeros(num_classes, dtype=torch.long),
        )

        self.register_buffer(
            "finding_momentum",
            torch.tensor(FINDING_PROTO_MOMENTUM, dtype=torch.float32),
        )
        self.register_buffer(
            "birads_momentum",
            torch.tensor(BIRADS_PROTO_MOMENTUM, dtype=torch.float32),
        )

    @torch.no_grad()
    def update_finding_prototypes(self, emb_f, finding_vec):
        for f in range(self.num_findings):
            mask = finding_vec[:, f] > 0.5

            if mask.sum() == 0:
                continue

            batch_proto = F.normalize(emb_f[mask].mean(dim=0), dim=0)
            m = self.finding_momentum[f].item()

            if self.proto_finding_updates[f] == 0:
                self.proto_finding[f] = batch_proto
            else:
                self.proto_finding[f] = F.normalize(
                    m * self.proto_finding[f] + (1.0 - m) * batch_proto,
                    dim=0,
                )

            self.proto_finding_updates[f] += 1

    @torch.no_grad()
    def update_birads_prototypes(self, emb_b, labels):
        labels = labels.long()

        for c in range(self.num_classes):
            mask = labels == c

            if mask.sum() == 0:
                continue

            batch_proto = F.normalize(emb_b[mask].mean(dim=0), dim=0)
            m = self.birads_momentum[c].item()

            if self.proto_birads_updates[c] == 0:
                self.proto_birads[c] = batch_proto
            else:
                self.proto_birads[c] = F.normalize(
                    m * self.proto_birads[c] + (1.0 - m) * batch_proto,
                    dim=0,
                )

            self.proto_birads_updates[c] += 1

    def forward_encoder(self, x, return_embeddings=False):
        return self.encoder(x, return_embeddings=return_embeddings)


@torch.no_grad()
def validate(model, loader, device, binary_loss_fn, class_names=None):
    class_names = class_names or BINARY_CLASS_NAMES
    model.eval()

    total_loss = 0.0
    preds_all = []
    targets_all = []

    for batch in loader:
        imgs = batch["qry_im"].to(device, non_blocking=True)
        binary_gt = batch["binary"].to(device, non_blocking=True).long()

        logits = model.forward_encoder(imgs, return_embeddings=False)
        loss = binary_loss_fn(logits, binary_gt)

        total_loss += loss.item()
        preds = logits.argmax(1)

        preds_all.extend(preds.cpu().numpy())
        targets_all.extend(binary_gt.cpu().numpy())

    acc = accuracy_score(targets_all, preds_all)
    macro_f1 = f1_score(targets_all, preds_all, average="macro", zero_division=0)
    weighted_f1 = f1_score(targets_all, preds_all, average="weighted", zero_division=0)
    cm = confusion_matrix(targets_all, preds_all, labels=[0, 1])

    print("\n--- Validation (Binary) ---")
    print(cm)
    print(
        classification_report(
            targets_all,
            preds_all,
            labels=[0, 1],
            target_names=class_names,
            zero_division=0,
        )
    )

    return {
        "loss": total_loss / max(len(loader), 1),
        "acc": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }


@torch.no_grad()
def test_model(model, loader, device, save_dir, name="Test", class_names=None):
    class_names = class_names or BINARY_CLASS_NAMES
    model.eval()

    preds_all = []
    targets_all = []

    for batch in loader:
        imgs = batch["qry_im"].to(device, non_blocking=True)
        binary_gt = batch["binary"].to(device, non_blocking=True).long()

        logits = model.forward_encoder(imgs, return_embeddings=False)
        preds = logits.argmax(1)

        preds_all.extend(preds.cpu().numpy())
        targets_all.extend(binary_gt.cpu().numpy())

    acc = accuracy_score(targets_all, preds_all)
    macro_f1 = f1_score(targets_all, preds_all, average="macro", zero_division=0)
    weighted_f1 = f1_score(targets_all, preds_all, average="weighted", zero_division=0)
    cm = confusion_matrix(targets_all, preds_all, labels=[0, 1])

    report = classification_report(
        targets_all,
        preds_all,
        labels=[0, 1],
        target_names=class_names,
        zero_division=0,
    )

    print(f"\n{'=' * 70}")
    print(f"{name} | Acc={acc:.4f} | Macro-F1={macro_f1:.4f} | Weighted-F1={weighted_f1:.4f}")
    print(cm)
    print(report)
    print("=" * 70)

    os.makedirs(save_dir, exist_ok=True)
    out_path = os.path.join(save_dir, f"{name.lower().replace(' ', '_')}_binary_report.txt")

    with open(out_path, "w") as fh:
        fh.write(f"{name}\n")
        fh.write(f"Acc={acc:.4f} | Macro-F1={macro_f1:.4f} | Weighted-F1={weighted_f1:.4f}\n\n")
        fh.write(str(cm) + "\n\n")
        fh.write(report)

    return {
        "acc": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }


def train_model(
    model,
    train_loader,
    val_loader,
    device,
    lr_backbone=5e-5,
    lr_heads=5e-5,
    epochs=60,
    patience=15,
    save_path="best_binary_dual_proto.pth",
    w_finding=0.5,
    w_birads=0.5,
    w_sep=0.5,
    sep_margin=0.3,
    warmup_epochs=3,
    ramp_epochs=5,
    class_names=None,
):
    class_names = class_names or BINARY_CLASS_NAMES
    os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
    log_path = save_path.replace(".pth", "_log.txt")

    binary_loss_fn = AsymmetricFocalLoss().to(device)

    proto_loss_fn_finding = MultiLabelProtoLoss(
        temperature=model.temperature,
        min_updates=PROTO_MIN_UPDATES,
    ).to(device)

    proto_loss_fn_birads = ProtoInfoNCELoss(
        temperature=model.temperature,
        min_updates=PROTO_MIN_UPDATES,
    ).to(device)

    sep_loss_fn = PairwiseSeparationLoss(
        margin=sep_margin,
        min_updates=PROTO_MIN_UPDATES,
    ).to(device)

    optimizer = AdamW(
        [
            {
                "params": list(model.encoder.features.parameters()) + list(model.encoder.avgpool.parameters()),
                "lr": lr_backbone,
                "weight_decay": 0.05,
            },
            {
                "params": model.encoder.binary_head.parameters(),
                "lr": lr_heads,
                "weight_decay": 0.05,
            },
            {
                "params": model.encoder.proj_head_finding.parameters(),
                "lr": lr_heads,
                "weight_decay": 0.05,
            },
            {
                "params": model.encoder.proj_head_birads.parameters(),
                "lr": lr_heads,
                "weight_decay": 0.05,
            },
        ]
    )

    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    best_macro_f1 = 0.0
    not_improved = 0

    for epoch in range(epochs):
        if epoch < warmup_epochs:
            pw_f = 0.0
            pw_b = 0.0
            pw_s = 0.0
        else:
            ramp = min(1.0, (epoch - warmup_epochs + 1) / max(ramp_epochs, 1))
            pw_f = w_finding * ramp
            pw_b = w_birads * ramp
            pw_s = w_sep * ramp

        model.train()

        e_binary = 0.0
        e_proto_f = 0.0
        e_proto_b = 0.0
        e_sep = 0.0

        preds_all = []
        targets_all = []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")

        for batch in pbar:
            imgs = batch["qry_im"].to(device, non_blocking=True)
            labels = batch["qry_gt"].to(device, non_blocking=True).long()
            binary_gt = batch["binary"].to(device, non_blocking=True).long()
            finding_vec = batch["finding_vec"].to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, enabled=(scaler is not None)):
                logits, emb_f, emb_b = model.forward_encoder(imgs, return_embeddings=True)

                binary_loss = binary_loss_fn(logits, binary_gt)

                if pw_f > 0:
                    proto_loss_f = proto_loss_fn_finding(
                        q=emb_f,
                        prototypes=model.proto_finding,
                        proto_updates=model.proto_finding_updates,
                        sample_weights_list=[FINDING_SAMPLE_WEIGHT[i] if i < len(FINDING_SAMPLE_WEIGHT) else 1.0 
                                            for i in range(imgs.shape[0])],
                        finding_vec=finding_vec,
                    )
                else:
                    proto_loss_f = torch.tensor(0.0, device=device)

                if pw_b > 0:
                    birads_pos_indices = labels.detach().cpu().tolist()
                    birads_sw = [BIRADS_SAMPLE_WEIGHT[int(lb)] for lb in birads_pos_indices]

                    proto_loss_b = proto_loss_fn_birads(
                        q=emb_b,
                        prototypes=model.proto_birads,
                        proto_updates=model.proto_birads_updates,
                        sample_weights_list=birads_sw,
                        pos_indices=birads_pos_indices,
                    )
                else:
                    proto_loss_b = torch.tensor(0.0, device=device)

                if pw_s > 0:
                    sep_loss = sep_loss_fn(
                        proto_finding=model.proto_finding,
                        proto_birads=model.proto_birads,
                        proto_finding_updates=model.proto_finding_updates,
                        proto_birads_updates=model.proto_birads_updates,
                    )
                else:
                    sep_loss = torch.tensor(0.0, device=device)

                total_loss = (
                    binary_loss
                    + pw_f * proto_loss_f
                    + pw_b * proto_loss_b
                    + pw_s * sep_loss
                )

            if scaler is not None:
                scaler.scale(total_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), 1.0)
                optimizer.step()

            with torch.no_grad():
                feat = model.encoder._extract(imgs)

                emb_f_fresh = F.normalize(
                    model.encoder.proj_head_finding(feat),
                    dim=1,
                )
                model.update_finding_prototypes(emb_f_fresh, finding_vec)

                emb_b_fresh = F.normalize(
                    model.encoder.proj_head_birads(feat),
                    dim=1,
                )
                model.update_birads_prototypes(emb_b_fresh, labels)

            preds = logits.argmax(1)
            preds_all.extend(preds.detach().cpu().numpy())
            targets_all.extend(binary_gt.detach().cpu().numpy())

            e_binary += binary_loss.item()
            e_proto_f += proto_loss_f.item()
            e_proto_b += proto_loss_b.item()
            e_sep += sep_loss.item()

            pbar.set_postfix(
                bin=f"{binary_loss.item():.3f}",
                pf=f"{proto_loss_f.item():.3f}",
                pb=f"{proto_loss_b.item():.3f}",
                sep=f"{sep_loss.item():.3f}",
                pwf=f"{pw_f:.2f}",
                pwb=f"{pw_b:.2f}",
                pws=f"{pw_s:.2f}",
            )

        n = max(len(train_loader), 1)

        train_acc = accuracy_score(targets_all, preds_all)
        train_macro_f1 = f1_score(targets_all, preds_all, average="macro", zero_division=0)
        train_wt_f1 = f1_score(targets_all, preds_all, average="weighted", zero_division=0)

        print(f"\n{'=' * 70}")
        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"bin={e_binary / n:.4f} | "
            f"proto_f={e_proto_f / n:.4f} | "
            f"proto_b={e_proto_b / n:.4f} | "
            f"sep={e_sep / n:.4f} | "
            f"pwf={pw_f:.3f} | "
            f"pwb={pw_b:.3f} | "
            f"pws={pw_s:.3f} | "
            f"train_acc={train_acc:.4f} | "
            f"train_macro_f1={train_macro_f1:.4f}"
        )

        print(
            f"Proto finding updates: {model.proto_finding_updates.cpu().tolist()} | "
            f"Proto birads updates: {model.proto_birads_updates.cpu().tolist()}"
        )

        print("\n--- Train (Binary) ---")
        print(
            classification_report(
                targets_all,
                preds_all,
                labels=[0, 1],
                target_names=class_names,
                zero_division=0,
            )
        )

        val_metrics = validate(
            model=model,
            loader=val_loader,
            device=device,
            binary_loss_fn=binary_loss_fn,
            class_names=class_names,
        )

        print(
            f"Val loss={val_metrics['loss']:.4f} | "
            f"Val acc={val_metrics['acc']:.4f} | "
            f"Val macro_f1={val_metrics['macro_f1']:.4f} | "
            f"Val weighted_f1={val_metrics['weighted_f1']:.4f}"
        )
        print("=" * 70)

        scheduler.step()

        with open(log_path, "a") as fh:
            fh.write(
                f"Epoch {epoch + 1} | "
                f"bin={e_binary / n:.4f} | "
                f"proto_f={e_proto_f / n:.4f} | "
                f"proto_b={e_proto_b / n:.4f} | "
                f"sep={e_sep / n:.4f} | "
                f"pwf={pw_f:.3f} | "
                f"pwb={pw_b:.3f} | "
                f"pws={pw_s:.3f} | "
                f"train_acc={train_acc:.4f} | "
                f"train_macro_f1={train_macro_f1:.4f} | "
                f"train_wt_f1={train_wt_f1:.4f} | "
                f"val_loss={val_metrics['loss']:.4f} | "
                f"val_acc={val_metrics['acc']:.4f} | "
                f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
                f"val_wt_f1={val_metrics['weighted_f1']:.4f}\n"
            )

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "best_macro_f1": best_macro_f1,
                    "config": {
                        "num_classes": NUM_CLASSES,
                        "num_findings": NUM_FINDINGS,
                        "w_finding": w_finding,
                        "w_birads": w_birads,
                        "w_sep": w_sep,
                        "sep_margin": sep_margin,
                        "temperature": model.temperature,
                        "class_names": class_names,
                    },
                },
                save_path,
            )

            print(f"Saved best model macro-F1={best_macro_f1:.4f}")
            not_improved = 0
        else:
            not_improved += 1

            if not_improved >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    return best_macro_f1


def run_experiments(train_loader, val_loader, test_loader, inbreast_loader):
    seed_everything(42)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    train_config = dict(
        lr_backbone=5e-5,
        lr_heads=5e-5,
        epochs=60,
        patience=15,
        w_finding=0.5,
        w_birads=0.5,
        w_sep=0.5,
        sep_margin=0.25,
        warmup_epochs=3,
        ramp_epochs=5,
        class_names=BINARY_CLASS_NAMES,
    )

    backbones = [

        {
            "name": "efficientnet_b3",
            "fn": models.efficientnet_b3,
            "w": models.EfficientNet_B3_Weights.DEFAULT,
        },
        {
            "name": "convnext_base",
            "fn": models.convnext_base,
            "w": models.ConvNeXt_Base_Weights.DEFAULT,
        },

    ]

    all_results = {}

    for cfg in backbones:
        out_dir = f"/workspace/Thesis_updated_results/binary_512_dual_proto/{cfg['name']}"
        save_path = f"{out_dir}/best_model.pth"

        os.makedirs(out_dir, exist_ok=True)

        print(f"\n{'#' * 70}")
        print(f"Backbone: {cfg['name']}")
        print("Task: Binary Classification + Finding Proto + BI-RADS Proto + Hierarchical Separation")
        print(f"Binary class weights: {BINARY_CLASS_WEIGHT}")
        print(f"Finding proto momentums: {FINDING_PROTO_MOMENTUM}")
        print(f"BI-RADS proto momentums: {BIRADS_PROTO_MOMENTUM}")
        print(f"Separation weight: {train_config['w_sep']}")
        print(f"Separation margin: {train_config['sep_margin']}")
        print(f"{'#' * 70}")

        model = DualProtoNet(
            backbone_name=cfg["name"],
            backbone_fn=cfg["fn"],
            backbone_weights=cfg["w"],
            emb_dim=128,
            dropout=0.2,
            num_classes=NUM_CLASSES,
            num_findings=NUM_FINDINGS,
            temperature=0.07,
        ).to(device)

        num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Trainable params: {num_params:,}")

        best_macro_f1 = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            save_path=save_path,
            **train_config,
        )

        ckpt = torch.load(save_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])

        print(f"Loaded epoch {ckpt['epoch'] + 1} | best macro-F1={ckpt['best_macro_f1']:.4f}")

        vindr_metrics = test_model(
            model=model,
            loader=test_loader,
            device=device,
            save_dir=out_dir,
            name="VinDr-Test",
            class_names=BINARY_CLASS_NAMES,
        )

        inbreast_metrics = test_model(
            model=model,
            loader=inbreast_loader,
            device=device,
            save_dir=out_dir,
            name="INbreast",
            class_names=BINARY_CLASS_NAMES,
        )

        all_results[cfg["name"]] = {
            "best_val_macro_f1": best_macro_f1,
            "vindr_acc": vindr_metrics["acc"],
            "vindr_macro_f1": vindr_metrics["macro_f1"],
            "vindr_weighted_f1": vindr_metrics["weighted_f1"],
            "inbreast_acc": inbreast_metrics["acc"],
            "inbreast_macro_f1": inbreast_metrics["macro_f1"],
            "inbreast_weighted_f1": inbreast_metrics["weighted_f1"],
        }

        del model, ckpt
        torch.cuda.empty_cache()
        gc.collect()

    print(f"\n{'=' * 95}")
    print("FINAL BINARY + DUAL PROTO + HIERARCHICAL SEPARATION RESULTS")
    print(f"{'=' * 95}")
    print(
        f"{'Model':<22} "
        f"{'Val Macro-F1':>14} "
        f"{'VinDr Macro-F1':>17} "
        f"{'INbreast Macro-F1':>20}"
    )
    print("-" * 95)

    for name, r in all_results.items():
        print(
            f"{name:<22} "
            f"{r['best_val_macro_f1']:>14.4f} "
            f"{r['vindr_macro_f1']:>17.4f} "
            f"{r['inbreast_macro_f1']:>20.4f}"
        )

    return all_results


if __name__ == "__main__":
    results = run_experiments(
        train_loader=tr_dl,
        val_loader=val_dl,
        test_loader=ts_dl,
        inbreast_loader=inbreast_dl,
    )

In [ ]:
rm -r /workspace/Thesis_updated_results/analysis_results_comprehensive

In [ ]:
"""
MammoDaViT — Mammography Dual Attention Vision Transformer
===========================================================
Derived from:
    DaViT: Dual Attention Vision Transformers
    Ding et al., ECCV 2022  — https://arxiv.org/abs/2204.03645

Original DaViT insight (transplanted here):
    Two orthogonal self-attention types run in ALTERNATING blocks:
      1. Spatial window attention  — spatial dimension as token scope
                                     captures local fine-grained texture
      2. Channel group attention   — channel dimension as token scope
                                     each token = entire-image abstract repr
                                     captures global interactions at O(C²) not O(N²)

What we change / add for mammography:
    ┌──────────────────────────────────────────────────────────────────────┐
    │  DaViT (original)           →  MammoDaViT (this file)               │
    ├──────────────────────────────────────────────────────────────────────┤
    │  Sequential blocks:          →  Fused block: spatial-window attn     │
    │  [spatial block, channel        channel-group attn, and DW-conv      │
    │   block] alternating            run in PARALLEL, merged via          │
    │                                 learned scalar gates (α, β, γ)       │
    ├──────────────────────────────────────────────────────────────────────┤
    │  Linear FFN (MLP)            →  Conv-FFN: 1×1 expand → 3×3 DW →     │
    │                                 1×1 project, preserves spatial bias  │
    ├──────────────────────────────────────────────────────────────────────┤
    │  Single-channel ImageNet     →  3-channel input: CC / MLO / diff     │
    │  (3ch via pretrain)             mammogram views encoded at stem      │
    ├──────────────────────────────────────────────────────────────────────┤
    │  Overlapping patch embed     →  Convolutional patch stem: stacked    │
    │  (7×7 stride-4 conv)            3×3 convs, preserves sub-lesion      │
    │                                 detail lost by large-stride embed    │
    ├──────────────────────────────────────────────────────────────────────┤
    │  Hierarchical 4 stages       →  Identical 4-stage hierarchy,         │
    │  C, 2C, 4C, 8C channels         same dims: 96, 192, 384, 768        │
    │  Patch merging downsampler      Patch merging unchanged              │
    ├──────────────────────────────────────────────────────────────────────┤
    │  Classification head         →  Focal-loss head (class imbalance)    │
    └──────────────────────────────────────────────────────────────────────┘

Input  : (B, 3, 512, 512)
Output : (B, num_classes)   default 3 — normal / benign / malignant
Params : ~28M  (matches DaViT-Tiny)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from einops import rearrange          # pip install einops


# ═══════════════════════════════════════════════════════════════════════════════
# 1.  CONVOLUTIONAL PATCH STEM
#     Why: DaViT uses a 7×7 stride-4 patch embed that discards fine spatial
#     detail.  Mammograms contain diagnostically critical micro-calcifications
#     (~0.1 mm).  Stacked 3×3 convs with cumulative stride-4 preserve these
#     while still compressing to a workable token grid.
# ═══════════════════════════════════════════════════════════════════════════════

class ConvStem(nn.Module):
    """
    3×512×512  →  96×128×128
    Three 3×3 convs (cumulative stride = 4) with BN + GELU.
    Overlapping receptive fields = no aliasing at boundaries.
    """
    def __init__(self, in_ch: int = 3, out_ch: int = 96):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch,      32,     3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.Conv2d(32,         64,     3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64,         out_ch, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ═══════════════════════════════════════════════════════════════════════════════
# 2.  PATCH MERGING  (unchanged from DaViT / Swin)
#     2× spatial downsampling, channel doubling.
# ═══════════════════════════════════════════════════════════════════════════════

class PatchMerging(nn.Module):
    def __init__(self, in_ch: int):
        super().__init__()
        self.norm = nn.LayerNorm(4 * in_ch)
        self.proj = nn.Linear(4 * in_ch, 2 * in_ch, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W)
        B, C, H, W = x.shape
        x = x.permute(0, 2, 3, 1)          # B H W C
        x0 = x[:, 0::2, 0::2, :]           # top-left
        x1 = x[:, 1::2, 0::2, :]           # bottom-left
        x2 = x[:, 0::2, 1::2, :]           # top-right
        x3 = x[:, 1::2, 1::2, :]           # bottom-right
        x = torch.cat([x0, x1, x2, x3], dim=-1)   # B H/2 W/2 4C
        x = self.norm(x)
        x = self.proj(x)                    # B H/2 W/2 2C
        return x.permute(0, 3, 1, 2)       # B 2C H/2 W/2


# ═══════════════════════════════════════════════════════════════════════════════
# 3.  SPATIAL WINDOW SELF-ATTENTION
#     Directly from DaViT §3.1 / Swin.
#     Partitions H×W into non-overlapping windows of size ws×ws.
#     Attention is performed locally inside each window → O(N·ws²) complexity.
#     Relative position bias added (learnable table) per DaViT.
# ═══════════════════════════════════════════════════════════════════════════════

class SpatialWindowAttention(nn.Module):
    """
    Args:
        dim        : channel dimension C
        num_heads  : attention heads
        window_size: local window size (ws).  DaViT uses 7 for ImageNet-224.
                     We use 8 → aligned to 512/64 = 8 at stage-1 (128px / 16).
    """
    def __init__(self, dim: int, num_heads: int = 4, window_size: int = 8):
        super().__init__()
        self.num_heads   = num_heads
        self.ws          = window_size
        self.head_dim    = dim // num_heads
        self.scale       = self.head_dim ** -0.5

        self.qkv  = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim,     bias=True)

        # Learnable relative position bias table  (2ws-1)² × H
        self.rel_pos_table = nn.Parameter(
            torch.zeros((2 * window_size - 1) ** 2, num_heads))
        nn.init.trunc_normal_(self.rel_pos_table, std=0.02)
        self._build_index(window_size)

    def _build_index(self, ws: int):
        coords = torch.stack(torch.meshgrid(
            torch.arange(ws), torch.arange(ws), indexing='ij'))   # 2, ws, ws
        coords = torch.flatten(coords, 1)                          # 2, ws²
        rel    = coords[:, :, None] - coords[:, None, :]           # 2, ws², ws²
        rel[0] += ws - 1
        rel[1] += ws - 1
        rel[0] *= 2 * ws - 1
        self.register_buffer('rel_idx', rel.sum(0))                # ws², ws²

    # ── window helpers ──────────────────────────────────────────────────────
    @staticmethod
    def _partition(x: torch.Tensor, ws: int) -> torch.Tensor:
        B, H, W, C = x.shape
        x = x.view(B, H // ws, ws, W // ws, ws, C)
        return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, ws * ws, C)

    @staticmethod
    def _reverse(windows: torch.Tensor, ws: int, H: int, W: int) -> torch.Tensor:
        B = int(windows.shape[0] / (H * W // ws ** 2))
        x = windows.view(B, H // ws, W // ws, ws, ws, -1)
        return x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        assert H % self.ws == 0 and W % self.ws == 0, \
            f"Spatial dims ({H},{W}) must be divisible by window_size {self.ws}"

        x = x.permute(0, 2, 3, 1)                      # B H W C
        windows = self._partition(x, self.ws)           # nWB, ws², C

        nWB, N, _ = windows.shape
        qkv = self.qkv(windows).reshape(nWB, N, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0) # each: nWB, H, ws², d

        attn = (q @ k.transpose(-2, -1)) * self.scale  # nWB, H, ws², ws²
        bias = self.rel_pos_table[self.rel_idx.view(-1)].view(
            self.ws ** 2, self.ws ** 2, self.num_heads).permute(2, 0, 1)
        attn = F.softmax(attn + bias.unsqueeze(0), dim=-1)

        out = (attn @ v).transpose(1, 2).reshape(nWB, N, C)
        out = self.proj(out)

        out = self._reverse(out, self.ws, H, W)         # B H W C
        return out.permute(0, 3, 1, 2)                  # B C H W


# ═══════════════════════════════════════════════════════════════════════════════
# 4.  CHANNEL GROUP SELF-ATTENTION  ← the key DaViT innovation
#
#     DaViT §3.2: "We apply self-attention to the TRANSPOSE of the token
#     matrix.  The channel dimension defines the token scope; the spatial
#     dimension defines the token feature dimension.  Each channel token
#     contains an abstract representation of the entire image."
#
#     Implementation detail from the paper:
#       • Group channels into G groups to keep complexity O(C·G) not O(C²)
#       • Within each group, every channel token attends to every other
#         channel token → captures global spatial info via the channel axis
#       • No positional encoding needed: channel tokens are permutation-
#         invariant w.r.t. spatial positions by construction
# ═══════════════════════════════════════════════════════════════════════════════

class ChannelGroupAttention(nn.Module):
    """
    Args:
        dim         : total channel dimension C
        num_groups  : G — number of channel groups.  Complexity = O(C * (C/G)).
                      DaViT default: G = C // 32  (groups of 32 channels each).
    """
    def __init__(self, dim: int, num_groups: int = None):
        super().__init__()
        # Default: groups of 32 channels each, matching DaViT
        self.G        = num_groups if num_groups is not None else max(1, dim // 32)
        self.group_ch = dim // self.G       # channels per group
        self.scale    = self.group_ch ** -0.5

        # Single-head per group (DaViT uses single-head channel attention)
        self.qkv  = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim,     bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        # Reshape: treat each channel as a token with feature dim = H*W
        # (B, C, H*W)  →  tokens of length C, each of dim H*W
        tokens = x.view(B, C, H * W)                   # B, C, HW

        # Apply QKV over the channel dimension
        # tokens transposed: (B, HW, C) for the linear
        t = tokens.transpose(1, 2)                      # B, HW, C
        qkv = self.qkv(t)                               # B, HW, 3C
        q, k, v = qkv.chunk(3, dim=-1)                  # each: B, HW, C

        # Re-group: reshape to (B*G, HW, group_ch)
        q = q.view(B, H * W, self.G, self.group_ch).permute(0, 2, 1, 3)
        k = k.view(B, H * W, self.G, self.group_ch).permute(0, 2, 1, 3)
        v = v.view(B, H * W, self.G, self.group_ch).permute(0, 2, 1, 3)
        # Now: (B, G, HW, group_ch) — attend over channel tokens

        # Swap HW and group_ch so we attend across channels, not spatial
        # per DaViT: q shape should be (B*G, group_ch, HW) for channel attention
        q = q.transpose(2, 3)   # B, G, group_ch, HW
        k = k.transpose(2, 3)
        v = v.transpose(2, 3)

        # Attention: (group_ch × group_ch) per group
        attn = (q @ k.transpose(-2, -1)) * self.scale  # B, G, group_ch, group_ch
        attn = F.softmax(attn, dim=-1)
        out  = attn @ v                                 # B, G, group_ch, HW

        # Reconstruct
        out = out.transpose(2, 3)                       # B, G, HW, group_ch
        out = out.permute(0, 2, 1, 3).contiguous()      # B, HW, G, group_ch
        out = out.view(B, H * W, C)                     # B, HW, C
        out = self.proj(out)                            # B, HW, C
        return out.transpose(1, 2).view(B, C, H, W)     # B, C, H, W


# ═══════════════════════════════════════════════════════════════════════════════
# 5.  CONVOLUTIONAL FFN
#     DaViT uses a standard 2-layer MLP as FFN.
#     We replace it with a conv-based bottleneck:
#       1×1 expand → 3×3 DW → GELU → 1×1 project
#     The 3×3 DW step re-injects spatial locality that the attention layers
#     may not capture for fine microcalcification texture in mammograms.
# ═══════════════════════════════════════════════════════════════════════════════

class ConvFFN(nn.Module):
    def __init__(self, dim: int, expand: int = 4):
        super().__init__()
        hidden = dim * expand
        self.net = nn.Sequential(
            nn.Conv2d(dim,    hidden, 1,             bias=False),
            nn.Conv2d(hidden, hidden, 3, padding=1,
                      groups=hidden,                 bias=False),
            nn.GELU(),
            nn.Conv2d(hidden, dim,    1,             bias=False),
        )
        self.norm = nn.GroupNorm(1, dim)   # LN equivalent for spatial tensors

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.net(self.norm(x))


# ═══════════════════════════════════════════════════════════════════════════════
# 6.  MAMMODAVIT BLOCK  ← the core novel unit
#
#     DaViT runs spatial-block → channel-block in alternation (sequential).
#     We fuse both into ONE block with a parallel third branch (DW-conv) and
#     merge via learned scalar gates:
#
#         y = LN(x)
#         s = SpatialWindowAtttn(y)        # local fine texture
#         c = ChannelGroupAttn(y)          # global abstract repr
#         t = DW-Conv 3×3(y)              # high-freq inductive bias
#         fused = α·s + β·c + γ·t         # learned gate per channel
#         x = x + fused                   # residual
#         x = ConvFFN(x)                  # spatial-aware mixing
#
#     Novelty over DaViT: the three branches run concurrently and their
#     contributions are dynamically balanced per-sample via scalar gates.
#     This means the network learns when to trust local texture vs global
#     context vs convolution — important because dense vs fatty breast tissue
#     changes which branch is most informative.
# ═══════════════════════════════════════════════════════════════════════════════

class MammoDaViTBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, window_size: int = 8,
                 ffn_expand: int = 4, num_groups: int = None, drop: float = 0.0):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)

        # Branch 1 — Spatial window attention (from DaViT, unchanged)
        self.spatial_attn  = SpatialWindowAttention(dim, num_heads, window_size)

        # Branch 2 — Channel group attention (from DaViT, unchanged)
        self.channel_attn  = ChannelGroupAttention(dim, num_groups)

        # Branch 3 — DW-conv  (new; provides spatial inductive bias)
        self.dw_conv = nn.Sequential(
            nn.Conv2d(dim, dim, 3, padding=1, groups=dim, bias=False),
            nn.BatchNorm2d(dim),
            nn.GELU(),
        )

        # Learned scalar gates — initialized to 1/3 each (equal contribution)
        self.alpha = nn.Parameter(torch.ones(1) / 3)   # spatial weight
        self.beta  = nn.Parameter(torch.ones(1) / 3)   # channel weight
        self.gamma = nn.Parameter(torch.ones(1) / 3)   # conv weight

        self.drop = nn.Dropout(drop)
        self.ffn  = ConvFFN(dim, ffn_expand)

    def _apply_norm(self, x: torch.Tensor) -> torch.Tensor:
        """Apply LayerNorm on a spatial tensor (B, C, H, W)."""
        B, C, H, W = x.shape
        return self.norm1(x.flatten(2).transpose(1, 2)).transpose(1, 2).view(B, C, H, W)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normed = self._apply_norm(x)

        s = self.spatial_attn(normed)      # (B, C, H, W)
        c = self.channel_attn(normed)      # (B, C, H, W)
        t = self.dw_conv(normed)           # (B, C, H, W)

        fused = self.alpha * s + self.beta * c + self.gamma * t
        x = x + self.drop(fused)
        x = self.ffn(x)
        return x


# ═══════════════════════════════════════════════════════════════════════════════
# 7.  FULL MammoDaViT MODEL
#
#     Hierarchy identical to DaViT-Tiny:
#       Stage 1: dim= 96, depth=1, heads=3,  window=8
#       Stage 2: dim=192, depth=1, heads=6,  window=8
#       Stage 3: dim=384, depth=3, heads=12, window=8
#       Stage 4: dim=768, depth=1, heads=24, window=8
#     (At 16×16 spatial the full attention window covers the entire map)
# ═══════════════════════════════════════════════════════════════════════════════

class MammoDaViT(nn.Module):
    """
    Args:
        in_channels : 3 for CC / MLO / difference-view stacking
        num_classes : 3  (normal, benign, malignant) — change for your task
        base_dim    : 96  — matches DaViT-Tiny
        depths      : blocks per stage
        num_heads   : attention heads per stage
        window_size : spatial window size (must divide all stage spatial dims)
        drop_rate   : dropout in classification head
    """
    def __init__(
        self,
        in_channels : int  = 3,
        num_classes : int  = 3,
        base_dim    : int  = 96,
        depths      : tuple = (1, 1, 3, 1),
        num_heads   : tuple = (3, 6, 12, 24),
        window_size : int  = 8,
        drop_rate   : float = 0.3,
    ):
        super().__init__()
        dims = [base_dim * (2 ** i) for i in range(4)]   # 96, 192, 384, 768

        # Stem
        self.stem = ConvStem(in_channels, dims[0])

        # Stages + Downsampling
        self.stages = nn.ModuleList()
        self.merges = nn.ModuleList()

        for i, (dim, depth, heads) in enumerate(zip(dims, depths, num_heads)):
            stage = nn.Sequential(*[
                MammoDaViTBlock(dim, heads, window_size)
                for _ in range(depth)
            ])
            self.stages.append(stage)
            if i < 3:
                self.merges.append(PatchMerging(dim))

        # Head
        self.norm = nn.LayerNorm(dims[-1])
        self.head = nn.Sequential(
            nn.Dropout(drop_rate),
            nn.Linear(dims[-1], num_classes),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.LayerNorm, nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        """Returns final spatial feature map — use for Grad-CAM."""
        x = self.stem(x)                        # B, 96,  128, 128
        for i, stage in enumerate(self.stages):
            x = stage(x)
            if i < len(self.merges):
                x = self.merges[i](x)
        return x                                # B, 768, 16,  16

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.forward_features(x)            # B, 768, 16, 16
        x = x.mean(dim=(2, 3))                  # B, 768  — global avg pool
        x = self.norm(x)
        return self.head(x)


# ═══════════════════════════════════════════════════════════════════════════════
# 8.  FOCAL LOSS
#     Mammography datasets are severely imbalanced.
#     Focal loss (Lin et al., RetinaNet) down-weights easy well-classified
#     examples so training focuses on hard-to-detect malignancies.
# ═══════════════════════════════════════════════════════════════════════════════

class FocalLoss(nn.Module):
    """
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    alpha : per-class weight tensor, shape (num_classes,)
    gamma : focusing parameter (2.0 recommended)
    """
    def __init__(self, gamma: float = 2.0, alpha: list = None, num_classes: int = 3):
        super().__init__()
        self.gamma = gamma
        if alpha is None:
            alpha = [1.0] * num_classes
        self.register_buffer('alpha', torch.tensor(alpha, dtype=torch.float32))

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce   = F.cross_entropy(logits, targets, reduction='none')
        p_t  = torch.exp(-ce)
        a_t  = self.alpha[targets]
        loss = a_t * (1 - p_t) ** self.gamma * ce
        return loss.mean()


# ═══════════════════════════════════════════════════════════════════════════════
# 9.  SMOKE TEST
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':
    model = MammoDaViT(in_channels=3, num_classes=3)

    total    = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params     : {total / 1e6:.2f}M")
    print(f"Trainable        : {trainable / 1e6:.2f}M")

    x = torch.randn(2, 3, 512, 512)
    feats  = model.forward_features(x)
    logits = model(x)

    print(f"Input            : {x.shape}")
    print(f"Feature map      : {feats.shape}")     # (2, 768, 16, 16)
    print(f"Logits           : {logits.shape}")    # (2, 3)

    criterion = FocalLoss(gamma=2.0, alpha=[0.25, 0.35, 0.40], num_classes=3)
    labels = torch.tensor([0, 2])
    loss   = criterion(logits, labels)
    print(f"Focal loss       : {loss.item():.4f}")
    print("Smoke test passed.")

    # Verify learned gate values (should start near 0.333 each)
    blk = model.stages[0][0]
    print(f"\nStage-0 block gates:")
    print(f"  alpha (spatial) = {blk.alpha.item():.4f}")
    print(f"  beta  (channel) = {blk.beta.item():.4f}")
    print(f"  gamma (conv)    = {blk.gamma.item():.4f}")